# Louvain 약물 후보 추천 – 결과와 한계 분석

질병 그래프를 Louvain으로 그룹으로 나누고 같은 그룹 질병의 약물을 추천했더니, 상위 10개 추천에서 가려 둔 적응증의 약 11%(Recall@10 10.97%)를 찾았다. 같은 분할에서 인기 약물 추천은 6.73%, 팀원의 Strict CBF는 18.26%다.

- 방법: 질병–유전자 유사도로 질병 그래프를 만들고 Louvain으로 질병 그룹을 찾는다. 대상 질병과 같은 그룹에 있는 다른 질병의 약물을 추천한다.
- 평가: 팀원 CBF와 같은 규칙으로 적응증을 학습·검증·테스트로 나누고(시드 42~46), 가려 둔 적응증을 상위 10개 추천에서 얼마나 찾는지 본다.
- 한계: 성능 차이를 단계별로 나누어 보면 대부분(약 70%)은 같은 그룹 질병에 한 표씩 주는 점수 방식에서 온다. 그룹 경계 자체의 영향은 작다(약 13%). 대상 질병과 가장 비슷한 학습 질병 50개 중 같은 그룹에 있는 것은 평균 50.6%뿐이지만, 이 비율은 성능 차이와 거의 관련이 없었다.
- 개선: 점수를 유사도 가중합으로 바꿨다. Recall@10이 10.97%에서 16.35%로 올랐다. 그래도 CBF(18.26%)와 PPR(29.74%)에는 못 미친다.
- 정리: 같은 설정에서는 그룹 제한이 없는 직접 이웃(CBF 방식)이 그룹 안 이웃보다 조금 높았다(검증 NDCG 0.1453 대 0.1393). 그룹은 같은 유사도 그래프에서 만들어지므로 새 정보를 더하지 못하고 이웃만 제한한다. 이 데이터에서 Louvain 그룹을 쓰는 이점은 확인하지 못했다.

**해석 범위:** 예비 분할에서 얻은 탐색 결과다. 적응증 58,174쌍에는 적응증이면서 금기(그 질병에 써서는 안 되는 약으로 기록된 관계)로도 기록된 339쌍이 남아 있고, 다섯 분할의 검증·테스트 관계가 서로 겹친다. 이미 여러 번 확인한 테스트로 새 변형을 평가했으므로 최종 성능으로 쓰지 않는다. 알려진 적응증을 다시 찾는 평가이므로 치료 효과나 안전성을 뜻하지 않는다.

## 1. 이 코드는 무엇을 하는가

Louvain은 그래프에서 연결이 촘촘한 노드 묶음을 찾는 군집 알고리즘이다. 이 노트북에서는 노드가 질병이고, 각 질병은 하나의 그룹에만 속한다.

1. 질병–유전자 연관 점수(`evidence_score`)가 0.3 이상인 유전자를 질병의 특징으로 쓴다.
2. 질병끼리 유전자 집합의 Jaccard 유사도를 구하고, 질병마다 가장 비슷한 k개를 이어 그래프를 만든다.
3. 그래프에 Louvain을 적용해 질병을 그룹으로 나눈다. 연결 강도에 대한 해상도(resolution)가 클수록 그룹이 작아진다.
4. 대상 질병의 약물 점수는 같은 그룹의 다른 질병 중 그 약물을 학습 관계로 가진 질병의 수다. 대상 질병에 이미 알려진 약물은 뺀다.
5. 점수가 높은 약물 10개를 추천한다. 점수가 있는 약물이 없으면 추천하지 않고 그 질병은 0점으로 센다. 인기 약물로 채우지 않는다.

평가는 질병별 Recall@10(숨긴 정답 중 상위 10개에 든 비율)과 이진 관련성 NDCG@10을 구해 질병 수로 평균한다. 정답은 데이터에 기록된 적응증이며, 임상 단계의 관계도 포함하므로 모두 승인된 치료라는 뜻은 아니다.

## 2. 팀 예비 분할에서의 성능

팀원 CBF 노트북의 규칙(`blake2b(seed|disease_id|drug_id)`)으로 적응증을 질병별로 나눈다. 전체 질병은 2,327개, 적응증은 58,174쌍이다. 학습·검증·테스트는 시드마다 35,287 / 11,350 / 11,537쌍이고 테스트 질병은 1,662개다. 이웃 수 10과 해상도 1.5는 다섯 검증 분할의 NDCG@10 평균으로 골랐다. 테스트는 학습+검증으로 다시 계산하고, 그 데이터에 나온 약물만 후보로 둔다.

CBF와 PPR 값은 팀원 자료에 적힌 수치이며 여기서 다시 실행한 것이 아니다. 팀원 원본 분할 파일과의 해시 대조도 하지 못했다. 인기 약물의 평균 Recall@10은 이번 실행에서 0.06728738로 팀원 PDF의 0.067287과 일치했다.

In [1]:
team_results = [
    ("인기 약물", 6.728738, 0.6480),
    ("Louvain", 10.971218, 0.2861),
    ("Strict CBF (팀원 PDF)", 18.2615, 0.1708),
    ("기본 PPR (팀원 노트북 추가 비교)", 29.74, 0.61),
    ("조절 PPR (팀원 노트북 추가 비교)", 30.75, 0.55),
]
for name, avg, sd in team_results:
    print(f"{name}: Recall@10 = {avg:.2f}% ± {sd:.2f}%p")

인기 약물: Recall@10 = 6.73% ± 0.65%p
Louvain: Recall@10 = 10.97% ± 0.29%p
Strict CBF (팀원 PDF): Recall@10 = 18.26% ± 0.17%p
기본 PPR (팀원 노트북 추가 비교): Recall@10 = 29.74% ± 0.61%p
조절 PPR (팀원 노트북 추가 비교): Recall@10 = 30.75% ± 0.55%p


Louvain의 평균 NDCG@10은 0.1152이고, 그룹 안에서 추천을 만들 수 있었던 질병은 85.88%다. 테스트 질병 1,662개 중 유전자 정보가 없거나 추천할 약물이 없는 질병은 0점으로 넣었다. Louvain은 인기 약물보다는 높지만 CBF와 PPR보다 낮다.

한계도 있다. 다섯 분할의 검증 평균으로 설정을 고른 뒤 각 테스트를 평가하는데, 한 분할의 테스트 관계가 다른 분할의 검증에 들어간다. 시드 42의 테스트 11,537쌍 중 6,725쌍이 다른 시드의 검증에 포함된다.

## 3. 한계 분석: 성능 차이는 어디서 오는가

기존 Louvain(검증 NDCG@10 0.0980)과 CBF 직접 이웃(0.1453)의 차이는 0.0473이다. 이 차이를 한 번에 하나씩 바꿔 가며 나누었다. 단계마다 다른 실험의 검증 결과(다섯 분할 평균)이고 순서에 따라 기여가 달라질 수 있어서, 대략적인 크기로만 본다.

| 단계 | 검증 NDCG@10 | 변화 | 전체 차이에서 차지하는 몫 |
|---|---:|---:|---:|
| 기존 Louvain: Jaccard, 같은 그룹 질병에 한 표씩 | 0.0980 | | |
| + 유사도로 가중(Jaccard 그래프 그대로) | 0.1310 | +0.0330 | 약 70% |
| + 유전자 점수 코사인(그래프도 다시 구성) | 0.1356 | +0.0046 | 약 10% |
| + IDF, 유사도 제곱 | 0.1389 | +0.0033 | 약 7% |
| + 그룹 제한 제거(직접 이웃 50개, CBF) | 0.1453 | +0.0064 | 약 13% |

각 단계의 설명은 4장에 있다. 가장 큰 몫은 그룹 경계가 아니라 점수 방식이다. 그룹을 유지한 채 가중만 바꿔도 격차의 70%가 줄었다. 그룹 경계를 없애서 얻는 이득은 마지막 0.0064다. 그래서 아래에서는 그룹 경계가 실제로 얼마나 영향을 주는지를 따로 본다.

### 3-1. 비슷한 질병이 그룹 밖에 있다

시드 42의 검증 질병 1,475개 중 학습에 나온 다른 질병과 유사도가 양수인 1,285개를 대상으로 했다. 나머지 190개는 아래 비율 평균에서 빼고, 추천 평가에서는 0점으로 둔다. 질병마다 가장 비슷한 학습 질병을 최대 50개 찾고, 그중 같은 Louvain 그룹에 있는 비율을 구했다.

| 관찰 | 시드 42 결과 |
|---|---:|
| 비슷한 질병 중 같은 그룹에 있는 비율: 질병별 평균 | **50.6%** |
| 같은 비율의 중앙값 | **46.0%** |
| 비슷한 질병 중 같은 그룹 / 다른 그룹 수: 질병별 평균 | **25.1개 / 24.0개** |
| 비슷한 질병 중 하나 이상이 그룹 밖에 있는 질병 | **1,142 / 1,285개** |
| 대상 질병이 속한 그룹 크기의 중앙값 | **110개** |

평균은 질병마다 비율을 구한 뒤 평균한 값이다. 모든 질병에 비슷한 질병이 정확히 50개 있는 것은 아니다. 1,142개는 비슷한 질병이 그룹 밖에도 있다는 뜻이며, 그 질병의 추천이 실패했다는 뜻은 아니다. 시드 42~46에서 값이 모두 같은데, 그래프와 이웃 후보가 되는 학습 질병 집합이 시드에 따라 달라지지 않기 때문이다. 다섯 번의 독립된 반복 결과가 아니다.

### 3-1-1. 그룹 밖 비율은 성능 차이를 설명하는가

위 비율이 낮은 질병일수록 그룹 안 이웃이 직접 이웃보다 크게 불리하다면, 그룹 경계가 원인이라는 증거가 된다. 시드 42의 검증 질병 1,285개를 비율 구간으로 나눠 NDCG@10(상위 50개 이웃, 유사도 제곱)을 비교했다.

| 그룹 안에 남은 비율 | 질병 수 | 그룹 안 이웃 | 직접 이웃 | 차이(직접 − 그룹) |
|---|---:|---:|---:|---:|
| 0.0 이상 0.2 미만 | 205 | 0.1249 | 0.1329 | +0.0080 |
| 0.2 이상 0.4 미만 | 362 | 0.1355 | 0.1384 | +0.0030 |
| 0.4 이상 0.6 미만 | 212 | 0.1552 | 0.1579 | +0.0027 |
| 0.6 이상 0.8 미만 | 215 | 0.1711 | 0.1846 | +0.0135 |
| 0.8 이상 | 291 | 0.2224 | 0.2231 | +0.0007 |

전체 평균은 그룹 안 0.1627, 직접 이웃 0.1676으로 차이가 0.005다. 구간별 차이는 비율과 일정한 방향이 없고(스피어만 순위 상관 0.01), 그룹 크기와도 관련이 없다(−0.03). 직접 이웃이 이긴 질병 221개와 그룹이 이긴 질병 165개의 평균 그룹 안 비율은 0.47과 0.44로 비슷하다. 따라서 "비슷한 질병의 절반이 그룹 밖에 있다"는 사실이 성능 손실로 이어진다는 증거는 이 데이터에서 약하다. 그룹 밖 이웃이 대상 질병의 정답 약물을 거의 갖고 있지 않거나, 유사도가 낮아 점수에 크게 기여하지 못하는 것으로 추정한다. 이 부분은 확인하지 못했다. 계산 코드는 코드 12에 있다.

In [1]:
retention_by_seed = [{'seed': 42, 'validation_diseases': 1475, 'diseases_with_neighbors': 1285, 'no_neighbors': 190, 'mean_retained_fraction': 0.5059736130364645, 'median_retained_fraction': 0.46, 'mean_top50_inside': 25.0568093385214, 'mean_top50_outside': 24.032684824902724, 'some_outside': 1142, 'median_community_size': 110, 'direct_recall_wins': 120, 'group_recall_wins': 87, 'recall_ties': 1268, 'direct_only_hits': 170, 'group_only_hits': 156}, {'seed': 43, 'validation_diseases': 1475, 'diseases_with_neighbors': 1285, 'no_neighbors': 190, 'mean_retained_fraction': 0.5059736130364645, 'median_retained_fraction': 0.46, 'mean_top50_inside': 25.0568093385214, 'mean_top50_outside': 24.032684824902724, 'some_outside': 1142, 'median_community_size': 110, 'direct_recall_wins': 117, 'group_recall_wins': 68, 'recall_ties': 1290, 'direct_only_hits': 166, 'group_only_hits': 120}, {'seed': 44, 'validation_diseases': 1475, 'diseases_with_neighbors': 1285, 'no_neighbors': 190, 'mean_retained_fraction': 0.5059736130364645, 'median_retained_fraction': 0.46, 'mean_top50_inside': 25.0568093385214, 'mean_top50_outside': 24.032684824902724, 'some_outside': 1142, 'median_community_size': 110, 'direct_recall_wins': 105, 'group_recall_wins': 61, 'recall_ties': 1309, 'direct_only_hits': 159, 'group_only_hits': 109}, {'seed': 45, 'validation_diseases': 1475, 'diseases_with_neighbors': 1285, 'no_neighbors': 190, 'mean_retained_fraction': 0.5059736130364645, 'median_retained_fraction': 0.46, 'mean_top50_inside': 25.0568093385214, 'mean_top50_outside': 24.032684824902724, 'some_outside': 1142, 'median_community_size': 110, 'direct_recall_wins': 133, 'group_recall_wins': 64, 'recall_ties': 1278, 'direct_only_hits': 203, 'group_only_hits': 133}, {'seed': 46, 'validation_diseases': 1475, 'diseases_with_neighbors': 1285, 'no_neighbors': 190, 'mean_retained_fraction': 0.5059736130364645, 'median_retained_fraction': 0.46, 'mean_top50_inside': 25.0568093385214, 'mean_top50_outside': 24.032684824902724, 'some_outside': 1142, 'median_community_size': 110, 'direct_recall_wins': 103, 'group_recall_wins': 79, 'recall_ties': 1293, 'direct_only_hits': 155, 'group_only_hits': 144}]
print("시드  질병 수  이웃 있음  그룹 안 비율  직접 이웃 Recall 우세  그룹 Recall 우세  Recall 동률")
for r in retention_by_seed:
    print(f"{r['seed']:>4}  {r['validation_diseases']:>7}  {r['diseases_with_neighbors']:>8}  {r['mean_retained_fraction']*100:>10.1f}%  {r['direct_recall_wins']:>19}  {r['group_recall_wins']:>15}  {r['recall_ties']:>11}")

시드  질병 수  이웃 있음  그룹 안 비율  직접 이웃 Recall 우세  그룹 Recall 우세  Recall 동률
  42     1475      1285        50.6%                  120               87         1268
  43     1475      1285        50.6%                  117               68         1290
  44     1475      1285        50.6%                  105               61         1309
  45     1475      1285        50.6%                  133               64         1278
  46     1475      1285        50.6%                  103               79         1293


### 3-2. 같은 질병, 같은 이웃 수로 비교하면

두 방법은 같은 특징, 유사도 제곱, 후보, 정답을 쓰고 학습 질병을 최대 50개 참고한다. 한 방법은 같은 그룹 안의 50개, 다른 방법은 그룹 제한 없이 가까운 50개를 쓴다. 약물은 상위 10개로 평가한다.

시드 42의 검증 질병 1,475개 중 직접 이웃의 Recall@10이 더 높은 질병은 **120개**, 그룹 안 이웃이 더 높은 질병은 **87개**, 같은 질병은 **1,268개**다. 같은 Recall이어도 맞힌 약물이나 순위가 같다는 뜻은 아니다. 질병별 정답 합계로는 직접 이웃만 맞힌 정답이 **170건**, 그룹 안 이웃만 맞힌 정답이 **156건**이다. 같은 약물이 다른 질병의 정답이면 따로 센다. 그룹 안 이웃이 더 잘 맞히는 경우도 있어서, 이 집계만으로 그룹 경계의 인과적 효과를 말할 수는 없다.

### 3-3. 실제 사례

시드 42에서 **B-cell acute lymphoblastic leukemia**(`EFO_0000094`)의 숨겨 둔 적응증 **LEUCOVORIN**(`CHEMBL1679`)은 직접 이웃 추천에서 **9위**, 같은 그룹 안 상위 50개 추천에서 **13위**였다. 직접 이웃 방식에서 이 약물의 점수에 기여한 질병 11개 중에는 다른 그룹의 **biliary tract neoplasm**(`EFO_0003891`, 코사인 유사도 0.1364)이 있었다.

순위 차이가 난 사례를 평가 뒤에 골랐고, 그룹 밖 질병 하나가 순위 차이의 유일한 원인이라는 증거는 아니다.

### 3-4. 그룹 안 이웃 수, 그룹 밖 이웃 추가, 약물 빈도

검증 분할만 사용했고 테스트 파일은 읽지 않았다. 질병 특징은 `연관 점수 × IDF`, 유사도는 코사인의 제곱이며, 팀원 CBF의 선택 설정과 같다. Louvain 그래프는 이웃 10, 해상도 1.0이다. 이 설정에서 해상도 1.0과 1.5를 검증했을 때 1.0의 검증 NDCG가 가장 높았다. IDF는 전체 질병–유전자 파일의 유전자별 질병 수로 구한다.

- 그룹 안 상위 K: 같은 그룹의 학습 질병 중 대상과 가장 비슷한 K개의 약물을 합산한다.
- 직접 이웃 상위 K: 그룹과 관계없이 대상과 가장 비슷한 학습 질병 K개의 약물을 합산한다.
- 그룹 밖 이웃 추가: 그룹 안 상위 50개의 점수에 그룹 밖 상위 20개의 점수를 0.5배로 더한다. 별도 정규화는 하지 않았다.
- 인기 약물 감점: 그룹 안 50개 방식의 약물 점수를 `(학습 빈도 + 1)^0.5`로 나눈다. 강도 0.5 한 가지만 시험했다.

표는 다섯 검증 분할의 질병별 평균이다. 여기서 이웃 K는 추천 약물 10개를 평가하는 @10과 다른 설정이다.

In [1]:
community_comparison_labels = ['그룹 안 상위 10', '그룹 안 상위 20', '그룹 안 상위 50', 'CBF 직접 이웃 10', 'CBF 직접 이웃 20', 'CBF 직접 이웃 50', '그룹 안 50 + 밖 20 (0.5배)', '그룹 안 50 + 인기 약물 감점']
community_comparison = [{'method': 'group_top_k', 'neighbor_k': '10', 'outside_k': '0', 'outside_weight': '0.0', 'popularity_beta': '0.0', 'n_splits': '5', 'mean_recall10': '0.13959902079386763', 'sd_recall10': '0.004003746073659312', 'mean_ndcg10': '0.13232474297968108', 'sd_ndcg10': '0.001873187419559914', 'mean_coverage': '0.8711864406779661'}, {'method': 'group_top_k', 'neighbor_k': '20', 'outside_k': '0', 'outside_weight': '0.0', 'popularity_beta': '0.0', 'n_splits': '5', 'mean_recall10': '0.14720369375912118', 'sd_recall10': '0.0023370856493744544', 'mean_ndcg10': '0.13826669830121155', 'sd_ndcg10': '0.0021916806113467554', 'mean_coverage': '0.8711864406779661'}, {'method': 'group_top_k', 'neighbor_k': '50', 'outside_k': '0', 'outside_weight': '0.0', 'popularity_beta': '0.0', 'n_splits': '5', 'mean_recall10': '0.14884346385298533', 'sd_recall10': '0.0017767211679512414', 'mean_ndcg10': '0.13926129229546283', 'sd_ndcg10': '0.0024796114764980962', 'mean_coverage': '0.8711864406779661'}, {'method': 'direct_top_k', 'neighbor_k': '10', 'outside_k': '0', 'outside_weight': '0.0', 'popularity_beta': '0.0', 'n_splits': '5', 'mean_recall10': '0.14396676088590907', 'sd_recall10': '0.0032620776657297916', 'mean_ndcg10': '0.13483505525732203', 'sd_ndcg10': '0.001417742390759111', 'mean_coverage': '0.8711864406779661'}, {'method': 'direct_top_k', 'neighbor_k': '20', 'outside_k': '0', 'outside_weight': '0.0', 'popularity_beta': '0.0', 'n_splits': '5', 'mean_recall10': '0.154532368558084', 'sd_recall10': '0.0022814898647942976', 'mean_ndcg10': '0.14285280547316068', 'sd_ndcg10': '0.002399644890800852', 'mean_coverage': '0.8711864406779661'}, {'method': 'direct_top_k', 'neighbor_k': '50', 'outside_k': '0', 'outside_weight': '0.0', 'popularity_beta': '0.0', 'n_splits': '5', 'mean_recall10': '0.1583195660030292', 'sd_recall10': '0.0025550380696712956', 'mean_ndcg10': '0.14527222541070417', 'sd_ndcg10': '0.0026824600876952714', 'mean_coverage': '0.8711864406779661'}, {'method': 'outside_added', 'neighbor_k': '50', 'outside_k': '20', 'outside_weight': '0.5', 'popularity_beta': '0.0', 'n_splits': '5', 'mean_recall10': '0.15691308423698558', 'sd_recall10': '0.002598686261156599', 'mean_ndcg10': '0.14510594036037314', 'sd_ndcg10': '0.0029579718927172393', 'mean_coverage': '0.8711864406779661'}, {'method': 'group_top_k_popularity_penalty', 'neighbor_k': '50', 'outside_k': '0', 'outside_weight': '0.0', 'popularity_beta': '0.5', 'n_splits': '5', 'mean_recall10': '0.10248612359884912', 'sd_recall10': '0.004917225406396142', 'mean_ndcg10': '0.08805999191592592', 'sd_ndcg10': '0.002744093049217208', 'mean_coverage': '0.8711864406779661'}]
print("방법                                         검증 Recall@10   검증 NDCG@10")
for label, r in zip(community_comparison_labels, community_comparison):
    print(f"{label:<34} {float(r['mean_recall10'])*100:>8.2f}%       {float(r['mean_ndcg10']):>8.4f}")

방법                                         검증 Recall@10   검증 NDCG@10
그룹 안 상위 10                            13.96%         0.1323
그룹 안 상위 20                            14.72%         0.1383
그룹 안 상위 50                            14.88%         0.1393
CBF 직접 이웃 10                          14.40%         0.1348
CBF 직접 이웃 20                          15.45%         0.1429
CBF 직접 이웃 50                          15.83%         0.1453
그룹 안 50 + 밖 20 (0.5배)                 15.69%         0.1451
그룹 안 50 + 인기 약물 감점                    10.25%         0.0881


같은 K에서는 직접 이웃이 그룹 안 이웃보다 높았다(K=50에서 NDCG 0.1453 대 0.1393). 그룹 안 50개에 그룹 밖 20개를 더하면 0.1451로 직접 이웃 50개에 가까워졌다. 그룹 제한이 점수를 낮추는 것은 맞지만 그 크기는 0.006 정도로 작다. 직접 이웃 50개의 값은 팀원의 선택 설정을 재현한 값(0.145272)과 일치했다. 인기 약물 감점은 0.0881로 크게 낮아져 이 설정에서는 도움이 되지 않았다.

이 결과는 해당 설정에 대한 관찰이며, 그룹 제한이나 빈도 보정이 항상 불리하다는 뜻은 아니다. 여러 조건을 같은 검증 집합에서 비교한 탐색이라 최고값을 최종 성능으로 쓸 수 없다. 이웃 10·20·50, 그룹 밖 10·20, 가중치 0.25·0.5의 전체 조합은 결과 CSV에 있다.

### 3-5. 그룹이 장점이 되지 못하는 이유

아래는 위 결과로 설명 가능한 해석이며, 직접 검증한 것은 아니다.

1. 그룹은 같은 유사도 그래프에서 만든다. 그래프의 이웃 정보에서 나온 것이어서 직접 이웃이 갖지 못한 정보를 더하지 못하고, 사용할 이웃을 줄이기만 한다.
2. 유사도로 가중하면 그룹 안에서도 덜 비슷한 질병은 이미 낮은 가중치를 받는다. 그룹으로 걸러서 줄일 잡음이 남아 있지 않다. 유사도를 쓰지 않는 균등 투표에서는 그룹이 유일한 필터라서 의미가 있지만, 그 방식이 가장 낮은 성능이었다.
3. 그룹이 크고 이질적이다. 중앙값 110개, 10~90% 구간은 48~202개다. 큰 그룹은 대상과 거의 닮지 않은 질병을 많이 포함한다.
4. 각 질병이 하나의 그룹에만 속한다. 경계에 있는 질병은 반대편의 가까운 질병을 쓰지 못한다. 다만 3-1-1에서 보았듯 이 손실의 크기는 작다.

그룹이 도움이 되는 조건을 확인하려면 유사도 값에 잡음이 많은 경우(유전자 수가 적은 질병)나 그룹 크기를 줄인 경우(해상도를 높인 경우)를 따로 비교해야 한다. 아직 하지 않았다.

## 4. 개선: 그룹 안 질병을 유사도로 가중한다

### 왜 개선되는가

기존 방식은 같은 그룹의 모든 질병이 약물에 한 표씩 준다. 그룹 크기의 중앙값이 110개라서, 대상 질병과 거의 닮지 않은 질병이 아주 비슷한 질병과 같은 영향력을 갖는다. 유사도를 가중치로 쓰면 비슷한 질병의 약물이 더 높은 점수를 받는다.

세 방식은 같은 CBF 예비 분할과 후보 정책을 쓰고, 이웃 수 10, 해상도 1.5, 그래프 시드 42로 고정했다. 다른 이웃 수나 해상도는 탐색하지 않았다. 약물 점수는 같은 그룹의 다른 질병에서만 계산하고 대상 질병의 학습 약물은 제외한다.

1. 기존 방식: 0.3 이상 유전자 집합의 Jaccard 유사도로 그래프를 만들고, 같은 그룹의 각 질병이 약물에 한 표씩 준다.
2. 유사도 가중: 같은 Jaccard 그래프와 그룹을 유지하고 `약물 점수 = Σ Jaccard(대상 질병, 같은 그룹의 다른 질병)`으로 바꾼다. 해당 약물을 학습 관계로 가진 질병만 합산한다.
3. 유전자 점수 + 유사도 가중: 0.3 이상의 `evidence_score`를 벡터 값으로 쓰고 코사인 유사도로 그래프와 그룹을 다시 만든다. `약물 점수 = Σ 코사인 유사도`로 계산한다. IDF는 쓰지 않는다. 특징 표현과 그룹 구조가 함께 바뀐다.

그래프의 이웃 수 10은 질병별 연결을 고르는 설정이다. 약물 점수에는 같은 그룹에서 대상과 유사도가 양수인 모든 질병을 합산하므로, 직접 이웃 10개만 쓰는 방식이 아니다.

### 결과

다섯 분할의 질병별 평균이다. 추천을 만들 수 없는 질병은 0점으로 넣었고 인기 약물로 채우지 않았다. 표준편차는 다섯 분할 사이의 값이다.

In [1]:
improvement_labels = {'original_jaccard_uniform': '기존: Jaccard + 동일 가중', 'jaccard_similarity_weighted': '유사도 가중: Jaccard', 'score_cosine_similarity_weighted': '유전자 점수 + 코사인 유사도 가중'}
validation_results = [{'phase': 'validation', 'method': 'original_jaccard_uniform', 'k': '10', 'recall_mean': '0.09829470910467256', 'recall_sd': '0.0030695046627882885', 'ndcg_mean': '0.09803309946529394', 'ndcg_sd': '0.004915953164199536', 'coverage_mean': '0.8711864406779661'}, {'phase': 'validation', 'method': 'jaccard_similarity_weighted', 'k': '10', 'recall_mean': '0.13979054440457733', 'recall_sd': '0.0028320758116279527', 'ndcg_mean': '0.13098058396851975', 'ndcg_sd': '0.003880661608351185', 'coverage_mean': '0.8711864406779661'}, {'phase': 'validation', 'method': 'score_cosine_similarity_weighted', 'k': '10', 'recall_mean': '0.14408071624544', 'recall_sd': '0.0024553486140622715', 'ndcg_mean': '0.1356133835978877', 'ndcg_sd': '0.0027300917933429517', 'coverage_mean': '0.8711864406779661'}]
test_results = [{'phase': 'test', 'method': 'original_jaccard_uniform', 'k': '10', 'recall_mean': '0.10971217563858765', 'recall_sd': '0.0028605301758861668', 'ndcg_mean': '0.11522583274956993', 'ndcg_sd': '0.002750535473736196', 'coverage_mean': '0.8588447653429603'}, {'phase': 'test', 'method': 'jaccard_similarity_weighted', 'k': '10', 'recall_mean': '0.16353356990799198', 'recall_sd': '0.002093831117689977', 'ndcg_mean': '0.16330762772205948', 'ndcg_sd': '0.0023053803892780053', 'coverage_mean': '0.8588447653429603'}, {'phase': 'test', 'method': 'score_cosine_similarity_weighted', 'k': '10', 'recall_mean': '0.16686652788468723', 'recall_sd': '0.00261854577647347', 'ndcg_mean': '0.16604177677572832', 'ndcg_sd': '0.002422366625492013', 'coverage_mean': '0.8592057761732852'}]
print("방법                                      검증 Recall@10   검증 NDCG@10   테스트 Recall@10   테스트 NDCG@10")
for v,t in zip(validation_results, test_results):
    print(f"{improvement_labels[t['method']]:<30} {float(v['recall_mean'])*100:>9.2f}%       {float(v['ndcg_mean']):>8.4f}       {float(t['recall_mean'])*100:>9.2f}%      {float(t['ndcg_mean']):>8.4f}")

방법                                      검증 Recall@10   검증 NDCG@10   테스트 Recall@10   테스트 NDCG@10
기존: Jaccard + 동일 가중                 9.83%         0.0980           10.97%        0.1152
유사도 가중: Jaccard                    13.98%         0.1310           16.35%        0.1633
유전자 점수 + 코사인 유사도 가중                14.41%         0.1356           16.69%        0.1660


Jaccard 그래프를 그대로 두고 그룹 안 유사도만 가중하자 테스트 Recall@10이 10.97%에서 16.35%로 올랐다. 유전자 점수를 쓰는 코사인 그래프에 유사도 가중을 함께 적용하면 16.69%였다. 코사인 방식은 그래프 구조도 달라지므로 점수 사용만의 효과로 볼 수 없다.

가중은 그룹 안의 덜 비슷한 질병의 영향을 줄일 뿐, 그룹 밖의 가까운 질병은 여전히 쓰지 못한다. 3장에서 본 손실이 남아 있기 때문에 CBF(18.26%)보다 낮은 것으로 추정한다. CBF와 PPR 값은 팀원 자료의 예비 수치다.

이 실험은 이미 여러 번 본 예비 테스트로 새 변형을 평가한 탐색이다. 적응증과 금기가 동시에 기록된 339쌍이 남아 있고 분할이 서로 겹치므로, 독립된 테스트에서도 같은 결과가 나오는지는 알 수 없다.

## 5. 추가 실험: 유전자 점수 기준과 적응증·금기 동시 기록 쌍

처음 구상한 전처리와 다른 선택지는 다음과 같다.

| 선택 항목 | 원래 설정과 이유 | 다른 선택과 유의점 |
|---|---|---|
| 질병–유전자 점수 | `evidence_score` 0.3 이상. 낮은 점수의 관계를 줄이고 비교 가능한 유전자 집합을 만든다. | 0.1 이상은 관계와 질병 범위를 늘리지만 약한 근거도 넣는다. 이 점수는 유전적 인과 확률이 아니다. |
| 질병 범위 | 유전자가 2,000개 이상인 포괄적 질병 범주 30개를 뺀다. 한 범주가 너무 많은 질병과 이어지는 영향을 줄이려는 선택이다. | 팀원 예비 실험처럼 포함할 수 있다. 평가 질병과 범위가 달라 직접 비교하면 안 된다. |
| 적응증·금기 동시 기록 쌍 | 같은 질병–약물 쌍이 적응증(치료에 쓴다)과 금기(써서는 안 된다)로 동시에 기록된 관계는 학습·평가에서 뺀다. | 출처마다 기록이 달라 생긴 충돌이며 실제 금기가 확정되는 것은 아니다. 원본은 보존하고 처리 규칙은 팀에서 정한다. |
| 유사도 | 0.3 이상 유전자 집합의 Jaccard. 점수의 존재만 써서 해석이 단순하다. | 점수 가중 코사인, IDF 등. 특징 사용 방식이 달라 별도 비교가 필요하다. |
| 그룹과 추천 | 비슷한 질병 그래프에 Louvain을 적용하고 그룹 안 약물 빈도로 순위를 매긴다. | 이웃 수, 해상도, 점수 계산 방식은 검증에서 고른다. |
| 평가 | 질병별 Recall@10, NDCG@10 평균. 추천 불가도 0점이고 인기 약물로 채우지 않는다. | 추천 가능 비율을 함께 본다. |

### 5-1. 실험 설계

**실험 A: 원래 데이터에서 점수와 동시 기록 쌍 비교.** 원래 질병 범위, 후보 약물 9,119개, 테스트 10,529쌍·1,597개 질병을 고정한다. 이 범위에서 적응증과 금기가 동시에 기록된 331쌍은 원래 학습·테스트에서 빠져 있다. 포함 조건에서는 이 331쌍을 학습에만 추가한다. 두 조건 모두 이 쌍은 추천에서 빼고 테스트 정답에도 넣지 않는다. 따라서 적응증과 금기가 동시에 기록된 관계를 학습 신호로 넣을 때의 변화만 본다. 전체 데이터의 겹친 쌍은 339쌍이고 8쌍은 이 질병 범위 밖이다. 점수 0.1/0.3과 포함/제외를 교차한 네 조건을 이웃 10, 해상도 1.5, 시드 42로 비교한다. 단일 분할이고 이미 설정 탐색에 쓴 테스트라 최종 성능이 아니다.

**실험 B: 팀원 CBF 예비 전처리.** 2장의 설정이다. 실험 A와 B는 데이터 범위, 정답, 후보 정책이 달라 수치를 직접 비교해 전처리 효과로 해석하면 안 된다.

### 5-2. 실험 A 결과

같은 테스트 질병에 대한 Strict Louvain 결과다.

In [1]:
# 실제 실행으로 저장한 결과다. 원본 CSV와 테스트 정답은 이 노트북에 포함하지 않는다.
experiment_a = [{'threshold': '0.3', 'conflict_policy': 'excluded', 'train_pairs': '43602', 'k': '10', 'evaluated_diseases': '1597', 'macro_recall': '0.12313255114091604', 'macro_ndcg': '0.12444399583747581', 'native_coverage': '0.9987476518472135'}, {'threshold': '0.3', 'conflict_policy': 'included_in_training', 'train_pairs': '43933', 'k': '10', 'evaluated_diseases': '1597', 'macro_recall': '0.12444834110260035', 'macro_ndcg': '0.12482778220195521', 'native_coverage': '0.9987476518472135'}, {'threshold': '0.1', 'conflict_policy': 'excluded', 'train_pairs': '43602', 'k': '10', 'evaluated_diseases': '1597', 'macro_recall': '0.1320545653105898', 'macro_ndcg': '0.12150906092034769', 'native_coverage': '1.0'}, {'threshold': '0.1', 'conflict_policy': 'included_in_training', 'train_pairs': '43933', 'k': '10', 'evaluated_diseases': '1597', 'macro_recall': '0.13196712253035417', 'macro_ndcg': '0.12162263965527605', 'native_coverage': '1.0'}]
print("유전자 기준   동시 기록 쌍 학습   학습 관계     Recall@10   NDCG@10   추천 가능 범위")
for r in experiment_a:
    policy = "제외" if r["conflict_policy"] == "excluded" else "포함"
    print(f"{r['threshold']:>9}   {policy:>13}   {int(r['train_pairs']):>8,}     {float(r['macro_recall'])*100:>8.2f}%   {float(r['macro_ndcg']):>8.4f}   {float(r['native_coverage'])*100:>8.2f}%")

유전자 기준   동시 기록 쌍 학습   학습 관계     Recall@10   NDCG@10   추천 가능 범위
      0.3              제외     43,602        12.31%     0.1244      99.87%
      0.3              포함     43,933        12.44%     0.1248      99.87%
      0.1              제외     43,602        13.21%     0.1215     100.00%
      0.1              포함     43,933        13.20%     0.1216     100.00%


점수 0.3에서 적응증과 금기가 동시에 기록된 331쌍을 학습에 추가하면 Recall이 약 0.13%p 높아지고, 0.1에서는 거의 변하지 않는다. 차이가 작고 단일 분할이라 포함하는 쪽이 낫다고 말하기 어렵다. 적응증과 금기가 동시에 기록된 관계는 신뢰하기 어려운 신호일 수 있어, 성능이 조금 올랐다는 이유만으로 포함할 근거는 되지 않는다.

0.1과 0.3의 차이는 유전자 정보를 쓰는 질병 그래프가 달라지는 데서 온다. 0.1에서는 같은 질병 범위에서 유전자가 없는 질병이 436개에서 267개로 줄고, 테스트 질병의 유전자 수 중앙값이 15개에서 42개로 늘었다. Recall은 0.1이 높고(13.21% 대 12.31%) NDCG는 0.3이 조금 높다(0.1244 대 0.1215). 질병 단위 부트스트랩으로 구한 Recall 차이의 95% 구간은 -0.23~+2.02%p로 0을 포함한다.

### 5-3. 유전자 수별 결과

점수 0.3, 적응증과 금기가 동시에 기록된 쌍을 제외한 조건이다. 추천이 없는 질병도 0점으로 넣었다. 집단마다 숨긴 정답 수와 질병 구성이 달라서, 유전자 수만으로 성능 차이를 설명할 수 없다.

In [1]:
gene_groups = [{'threshold': '0.3', 'grouping': 'threshold_specific', 'gene_count': '1', 'diseases': '167', 'native_count': '166', 'native_coverage': '0.9940119760479041', 'macro_recall_all': '0.12416595380667236', 'macro_ndcg_all': '0.09542074283163761', 'macro_recall_native': '0.12491394148020653', 'macro_ndcg_native': '0.09599556658363542'}, {'threshold': '0.3', 'grouping': 'threshold_specific', 'gene_count': '2-4', 'diseases': '281', 'native_count': '280', 'native_coverage': '0.99644128113879', 'macro_recall_all': '0.16247910789464065', 'macro_ndcg_all': '0.12008322912068432', 'macro_recall_native': '0.1630593904228358', 'macro_ndcg_native': '0.12051209779611534'}, {'threshold': '0.3', 'grouping': 'threshold_specific', 'gene_count': '5-9', 'diseases': '209', 'native_count': '209', 'native_coverage': '1.0', 'macro_recall_all': '0.10232150386415359', 'macro_ndcg_all': '0.08109993630121203', 'macro_recall_native': '0.10232150386415359', 'macro_ndcg_native': '0.08109993630121203'}, {'threshold': '0.3', 'grouping': 'threshold_specific', 'gene_count': '10+', 'diseases': '940', 'native_count': '940', 'native_coverage': '1.0', 'macro_recall_all': '0.11581398538332606', 'macro_ndcg_all': '0.1405409821592551', 'macro_recall_native': '0.11581398538332606', 'macro_ndcg_native': '0.1405409821592551'}]
print("유전자 수  질병 수  자체 추천 가능  Recall@10  NDCG@10")
for r in gene_groups:
    print(f"{r['gene_count']:>8}  {int(r['diseases']):>6}  {int(r['native_count']):>13}  {float(r['macro_recall_all'])*100:>8.2f}%  {float(r['macro_ndcg_all']):>7.4f}")

유전자 수  질병 수  자체 추천 가능  Recall@10  NDCG@10
       1     167            166     12.42%   0.0954
     2-4     281            280     16.25%   0.1201
     5-9     209            209     10.23%   0.0811
     10+     940            940     11.58%   0.1405


## 6. 한계와 다음 단계

- 알려진 적응증을 다시 찾는 평가라서 새 약물의 치료 효과를 말해 주지 않는다.
- 질병–유전자 점수에 이미 알려진 약물의 근거가 섞여 있을 수 있다. 이를 가려내지 못했다.
- 성능이 낮은 주된 원인은 같은 그룹 질병에 한 표씩 주는 점수 방식이다. 유사도 가중으로 이를 개선해도 CBF와 PPR에는 못 미친다. 그룹 경계의 영향은 작지만 0이 아니다(검증 NDCG 0.006).
- 그룹이 도움이 되는 조건(유전자가 적은 질병, 더 높은 해상도)은 확인하지 못했다.
- 각 질병은 하나의 그룹에만 속하고, 유전자 정보가 없는 질병은 추천을 만들 수 없다. 테스트 질병의 추천 가능 비율은 85.88%다.
- 예비 데이터에 적응증과 금기가 동시에 기록된 339쌍이 남아 있고, 다섯 분할이 서로 겹친다. 이미 본 테스트에서 새 변형을 평가했다.
- 팀이 정제한 공통 데이터와 고정 테스트를 정하면, 검증에서 설정을 고르고 테스트는 마지막에 한 번만 평가해야 한다.

# 재실행 코드

아래 코드에는 데이터 생성, 전처리, Louvain, 평가, 실험 A와 B, 개선 실험, 그룹 경계 분석이 모두 들어 있다. 데이터는 노트북에 포함하지 않는다. OptimusKG 입력 파일을 따로 준비하고 아래 경로를 수정한다.

필요 패키지는 `numpy`, `scipy`, `networkx`, `polars`다. 예: `%pip install "numpy>=2,<3" "scipy>=1.11,<2" "networkx>=3.2,<4" "polars>=1,<2"`. 팀의 환경에 맞춰 버전을 고정한다. 전체 재실행은 몇 분 이상 걸린다.

기본값 `RUN_EXPERIMENTS = False`는 저장된 결과만 보여 준다. 모든 코드 셀을 순서대로 실행한 뒤 마지막 셀에서 값을 `True`로 바꾸면, 원본을 바꾸지 않고 별도 작업 폴더에 결과를 만든다.

In [ ]:
from pathlib import Path
import sys

# 본인 환경의 경로로 바꾼다.
OPTIMUS_ROOT = Path("optimuskg")
WORK_DIR = Path.home() / "louvain_notebook_work"
SPLIT_DIR = OPTIMUS_ROOT / "split"
PROCESSED_DIR = OPTIMUS_ROOT / "processed"
RAW_PARQUET = OPTIMUS_ROOT / "raw" / "edges" / "disease_gene.parquet"
RUN_EXPERIMENTS = False

print("데이터 경로:", OPTIMUS_ROOT)
print("재실행:", RUN_EXPERIMENTS)

## 코드 1. 질병 그래프 만들기

질병–유전자 관계에서 질병 간 Jaccard 유사도를 구하고, 질병마다 가장 비슷한 k개를 이어 그래프를 만든다. 이 셀은 함수만 정의한다.

In [ ]:
"""Read a shared project split and build a weighted Jaccard disease graph."""
from __future__ import annotations

import csv
from pathlib import Path
import numpy as np
from scipy.sparse import coo_matrix
import networkx as nx


def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def build_similarity(data_dir: Path):
    disease_rows = read_csv(data_dir / "diseases.csv")
    diseases = sorted({row["disease_id"] for row in disease_rows})
    disease_index = {disease: i for i, disease in enumerate(diseases)}
    relations = read_csv(data_dir / "disease_gene.csv")
    pairs = sorted({(disease_index[row["disease_id"]], row["gene_id"]) for row in relations
                    if row["disease_id"] in disease_index and row["gene_id"]})
    gene_ids = {gene for _, gene in pairs}
    gene_index = {gene: i for i, gene in enumerate(sorted(gene_ids))}
    rows = np.fromiter((p[0] for p in pairs), dtype=np.int32, count=len(pairs))
    cols = np.fromiter((gene_index[p[1]] for p in pairs), dtype=np.int32, count=len(pairs))
    x = coo_matrix((np.ones(len(pairs), dtype=np.int32), (rows, cols)),
                   shape=(len(diseases), len(gene_ids))).tocsr()
    degrees = np.asarray(x.sum(axis=1)).ravel()
    intersections = (x @ x.T).tocsr()
    intersections.setdiag(0)
    intersections.eliminate_zeros()
    similarities = []
    for index in range(len(diseases)):
        row = intersections.getrow(index)
        union = degrees[index] + degrees[row.indices] - row.data
        valid = union > 0
        ids = row.indices[valid]
        weights = row.data[valid] / union[valid]
        # A stable ID tie break ensures reproducible nearest neighbours.
        order = np.lexsort((ids, -weights))
        similarities.append([(diseases[int(i)], float(w)) for i, w in zip(ids[order], weights[order])])
    return diseases, similarities, {
        "diseases": len(diseases), "genes": len(gene_ids), "unique_disease_gene_pairs": len(pairs),
        "diseases_without_genes": int(np.count_nonzero(degrees == 0)),
    }


def build_graph(diseases, similarities, k):
    graph = nx.Graph()
    graph.add_nodes_from(diseases)
    neighbors = {}
    for disease, row in zip(diseases, similarities):
        selected = row[:k]
        neighbors[disease] = selected
        for other, weight in selected:
            graph.add_edge(disease, other, weight=weight)
    return graph, neighbors

## 코드 2. 평가 지표

질병별 Recall@K와 NDCG@K를 구해 평균한다. 추천이 없는 질병은 0점으로 포함한다. 이 셀은 함수만 정의한다.

In [ ]:
"""Evaluate rankings with macro Recall@K and binary-relevance NDCG@K."""
from __future__ import annotations
from collections import defaultdict
import math


def group_pairs(rows):
    result = defaultdict(set)
    for row in rows:
        result[row["disease_id"]].add(row["drug_id"])
    return dict(result)


def metrics(rankings, truth, k, native_coverage=None):
    per_disease = []
    for disease in sorted(truth):
        relevant = truth[disease]
        if not relevant:
            continue
        ranked = rankings.get(disease, [])[:k]
        hits = [int(drug in relevant) for drug in ranked]
        recall = sum(hits) / len(relevant)
        dcg = sum(hit / math.log2(rank + 2) for rank, hit in enumerate(hits))
        ideal = sum(1 / math.log2(rank + 2) for rank in range(min(k, len(relevant))))
        per_disease.append({"disease_id": disease, "n_relevant": len(relevant),
                            "n_recommended": len(ranked), "hits": sum(hits),
                            "recall": recall, "ndcg": dcg / ideal if ideal else 0.0,
                            "native_recommendation": bool(native_coverage and native_coverage.get(disease))})
    n = len(per_disease)
    return {"k": k, "evaluated_diseases": n,
            "macro_recall": sum(row["recall"] for row in per_disease) / n if n else 0.0,
            "macro_ndcg": sum(row["ndcg"] for row in per_disease) / n if n else 0.0,
            "native_coverage": sum(row["native_recommendation"] for row in per_disease) / n if n else 0.0}, per_disease

## 코드 3. Louvain 추천과 기본 실험

검증에서 이웃 수와 해상도를 고르고 테스트를 한 번 평가한다. 같은 그룹 질병의 학습 약물 빈도로 추천한다. 이 셀은 함수만 정의한다.

In [ ]:
"""Select Louvain settings on a training-only validation set; evaluate once on test."""
from __future__ import annotations
import argparse
from collections import Counter, defaultdict
import csv
import hashlib
import json
from pathlib import Path
import random
import time
import networkx as nx



def checksum(path):
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def validation_split(train, seed, fraction=0.2):
    rng = random.Random(seed)
    keep, heldout = {}, {}
    for disease, drug_set in sorted(train.items()):
        drugs = sorted(drug_set)
        if len(drugs) < 2:
            keep[disease] = set(drugs)
            continue
        rng.shuffle(drugs)
        count = min(len(drugs) - 1, max(1, int(len(drugs) * fraction)))
        heldout[disease] = set(drugs[:count])
        keep[disease] = set(drugs[count:])
    return keep, heldout


def communities_for(graph, resolution, seed):
    if graph.number_of_edges() == 0:
        return [{disease} for disease in graph.nodes]
    return nx.community.louvain_communities(graph, weight="weight", resolution=resolution, seed=seed)


def make_rankings(method, queries, train, excluded, candidates, k, *, neighbors=None, communities=None, fallback=False):
    popularity = Counter(drug for drugs in train.values() for drug in drugs)
    popular_order = sorted(candidates, key=lambda d: (-popularity[d], d))
    community_by_disease = {}
    community_counts = []
    if method == "louvain":
        for index, community in enumerate(communities):
            community_counts.append(Counter(drug for disease in community for drug in train.get(disease, set())))
            for disease in community:
                community_by_disease[disease] = index
    rankings, native, score_maps = {}, {}, {}
    for disease in sorted(queries):
        known = train.get(disease, set())
        forbidden = known | excluded.get(disease, set())
        scores = Counter()
        if method == "louvain" and disease in community_by_disease:
            scores = community_counts[community_by_disease[disease]].copy()
            scores.subtract(known)  # votes must come from other diseases
        elif method == "neighbors":
            scores.update(drug for other, _ in neighbors.get(disease, []) for drug in train.get(other, set()))
        elif method == "popularity":
            scores = popularity.copy()
        selected = sorted((drug for drug, score in scores.items()
                           if score > 0 and drug in candidates and drug not in forbidden),
                          key=lambda drug: (-scores[drug], drug))[:k]
        native[disease] = bool(selected)
        native_set = set(selected)
        if fallback:
            for drug in popular_order:
                if len(selected) >= k:
                    break
                if drug not in forbidden and drug not in native_set and popularity[drug] > 0:
                    selected.append(drug)
                    native_set.add(drug)
        rankings[disease] = selected
        score_maps[disease] = {drug: (scores[drug] if scores[drug] > 0 else 0) for drug in selected}
    return rankings, native, score_maps


def write_csv(path, headers, rows):
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=headers)
        writer.writeheader()
        writer.writerows(rows)


def run_original_experiment():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-dir", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--top-k", type=int, default=10)
    parser.add_argument("--neighbors", type=int, nargs="+", default=[10, 20, 30])
    parser.add_argument("--resolutions", type=float, nargs="+", default=[0.5, 1.0, 1.5])
    args = parser.parse_args()
    if args.top_k < 1 or any(k < 1 for k in args.neighbors) or any(r <= 0 for r in args.resolutions):
        parser.error("top-k, neighbors and resolutions must be positive")
    args.output_dir.mkdir(parents=True, exist_ok=True)
    files = ["diseases.csv", "disease_gene.csv", "drugs.csv", "train.csv", "test.csv", "excluded.csv"]
    inputs = {name: checksum(args.data_dir / name) for name in files}
    train, test = group_pairs(read_csv(args.data_dir / "train.csv")), group_pairs(read_csv(args.data_dir / "test.csv"))
    excluded = group_pairs(read_csv(args.data_dir / "excluded.csv"))
    candidates = {row["drug_id"] for row in read_csv(args.data_dir / "drugs.csv")}
    for disease, truth in test.items():
        if truth & train.get(disease, set()) or truth & excluded.get(disease, set()):
            raise ValueError(f"Train/test/excluded overlap for {disease}")
        if not truth <= candidates:
            raise ValueError(f"Test includes drugs outside candidate set: {disease}")
    validation_train, validation_truth = validation_split(train, args.seed)
    if not validation_truth:
        raise ValueError("No disease has enough training positives for validation")
    start = time.perf_counter()
    diseases, similarities, data_stats = build_similarity(args.data_dir)
    val_rows = []
    for k in args.neighbors:
        graph, neighbors = build_graph(diseases, similarities, k)
        for resolution in args.resolutions:
            comms = communities_for(graph, resolution, args.seed)
            ranking, native, _ = make_rankings("louvain", validation_truth, validation_train,
                                               excluded, candidates, args.top_k,
                                               communities=comms, fallback=False)
            result, _ = metrics(ranking, validation_truth, args.top_k, native)
            result.update({"neighbors": k, "resolution": resolution, "communities": len(comms)})
            val_rows.append(result)
            print("VALIDATION", json.dumps(result), flush=True)
    best = sorted(val_rows, key=lambda x: (-x["macro_ndcg"], -x["macro_recall"], x["neighbors"], x["resolution"]))[0]
    graph, neighbors = build_graph(diseases, similarities, best["neighbors"])
    communities = communities_for(graph, best["resolution"], args.seed)
    results, predictions, details = {}, [], []
    for method in ("popularity", "neighbors", "louvain", "louvain_no_fallback"):
        ranking_method = "louvain" if method == "louvain_no_fallback" else method
        rankings, native, scores = make_rankings(ranking_method, test, train, excluded, candidates, args.top_k,
                                                neighbors=neighbors, communities=communities, fallback=method != "louvain_no_fallback")
        result, per_disease = metrics(rankings, test, args.top_k, native)
        results[method] = result
        details.extend({"method": method, **row} for row in per_disease)
        if method == "louvain":
            for disease, ranked in rankings.items():
                for rank, drug in enumerate(ranked, start=1):
                    predictions.append({"disease_id": disease, "drug_id": drug, "score": scores[disease][drug], "rank": rank})
        print("TEST", method, json.dumps(result), flush=True)
    summary = {"seed": args.seed, "validation_fraction": 0.2, "validation_fallback": False,
               "validation_diseases": len(validation_truth), "selection_metric": "macro_ndcg",
               "selected_parameters": {"neighbors": best["neighbors"], "resolution": best["resolution"]},
               "graph": {**data_stats, "edges": graph.number_of_edges(),
                         "isolated_diseases": len(list(nx.isolates(graph))), "communities": len(communities),
                         "singleton_communities": sum(len(c)==1 for c in communities),
                         "modularity": nx.community.modularity(graph, communities, weight="weight", resolution=best["resolution"]) if graph.number_of_edges() else None},
               "test": results, "elapsed_seconds": time.perf_counter()-start, "input_sha256": inputs,
               "note": "The overall evidence score may contain clinical information derived from held-out indications. Random edge-split results are not clinical validation."}
    write_csv(args.output_dir / "validation.csv", list(val_rows[0]), val_rows)
    write_csv(args.output_dir / "predictions.csv", ["disease_id", "drug_id", "score", "rank"], predictions)
    write_csv(args.output_dir / "per_disease.csv", ["method", "disease_id", "n_relevant", "n_recommended", "hits", "recall", "ndcg", "native_recommendation"], details)
    (args.output_dir / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print("DONE", json.dumps(summary, ensure_ascii=False), flush=True)

## 코드 4. 점수 0.1 데이터 만들기

원본 parquet에서 점수 0.1 이상의 질병–유전자 관계를 뽑는다. 0.3 이상 부분이 공유 분할과 일치하는지 확인한다. 이 셀은 함수만 정의한다.

In [ ]:
"""Rebuild a thresholded split from the original OptimusKG parquet.

This only writes a separate output directory; it does not alter the shared split.
"""
import argparse
from pathlib import Path
import shutil
import polars as pl


def prepare_threshold_data():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--raw', type=Path, required=True)
    parser.add_argument('--split', type=Path, required=True)
    parser.add_argument('--output', type=Path, required=True)
    parser.add_argument('--threshold', type=float, default=0.1)
    args = parser.parse_args()
    if not 0 <= args.threshold <= 1:
        parser.error('threshold must be between 0 and 1')
    args.output.mkdir(parents=True, exist_ok=True)
    disease_ids = pl.read_csv(args.split / 'diseases.csv').select('disease_id').unique()
    raw = pl.scan_parquet(args.raw)
    pairs = (raw.select(pl.col('from').alias('disease_id'), pl.col('to').alias('gene_id'),
                        pl.col('properties').struct.field('evidence_score').alias('evidence_score'))
             .filter(pl.col('evidence_score') >= args.threshold)
             .join(disease_ids.lazy(), on='disease_id', how='semi').collect())
    duplicate_count = pairs.height - pairs.unique(subset=['disease_id', 'gene_id']).height
    if duplicate_count:
        raise ValueError(f'Raw data contains {duplicate_count} duplicate disease–gene pairs')
    # Verify that threshold 0.3 exactly reproduces the input split, including scores.
    ref = pl.read_csv(args.split / 'disease_gene.csv').select('disease_id', 'gene_id', 'evidence_score')
    from_raw = pairs.filter(pl.col('evidence_score') >= 0.3)
    if args.threshold <= 0.3:
        keys = ['disease_id', 'gene_id']
        diff1 = ref.join(from_raw, on=keys, how='anti').height
        diff2 = from_raw.join(ref, on=keys, how='anti').height
        scores = ref.join(from_raw, on=keys, suffix='_raw')
        score_diff = scores.select((pl.col('evidence_score') - pl.col('evidence_score_raw')).abs().max()).item()
        if diff1 or diff2 or (score_diff is not None and score_diff > 1e-12):
            raise ValueError(f'Raw >= 0.3 and shared split differ: {diff1} / {diff2} rows, score diff {score_diff}')
    pairs.sort(['disease_id','gene_id']).write_csv(args.output / 'disease_gene.csv')
    for name in ['diseases.csv','drugs.csv','train.csv','test.csv','excluded.csv']:
        shutil.copy2(args.split / name, args.output / name)
    print(f'threshold={args.threshold} rows={pairs.height} diseases={pairs["disease_id"].n_unique()} verified_shared_t03=True', flush=True)

## 코드 5. 적응증·금기 동시 기록 쌍의 포함·제외 비교

같은 테스트 정답에서 적응증과 금기가 동시에 기록된 331쌍을 학습에 넣을 때와 뺄 때를 비교한다. 이 셀은 함수만 정의한다.

In [ ]:
"""Compare using versus withholding conflicting train edges on identical clean test labels."""
import argparse
from pathlib import Path
import json
from collections import Counter
from copy import deepcopy


def run_conflict_ablation():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument('--split',type=Path,required=True)
    p.add_argument('--t01',type=Path,required=True)
    p.add_argument('--indication',type=Path,required=True)
    p.add_argument('--output',type=Path,required=True)
    a=p.parse_args();a.output.mkdir(parents=True,exist_ok=True)
    base=group_pairs(read_csv(a.split/'train.csv')); truth=group_pairs(read_csv(a.split/'test.csv'))
    excluded=group_pairs(read_csv(a.split/'excluded.csv'))
    official={(r['disease_id'],r['drug_id']):r for r in read_csv(a.indication)}
    conflict={(d,t) for d,drugs in excluded.items() for t in drugs}
    if len(conflict)!=331 or not all(official[pair]['also_asserted_contraindication']=='True' for pair in conflict):
        raise ValueError('Expected 331 excluded contraindication conflicts in selected disease scope')
    if any(drugs & truth.get(d,set()) or drugs & base.get(d,set()) for d,drugs in excluded.items()):
        raise ValueError('Excluded conflicts overlap train or test')
    expanded=deepcopy(base)
    for d,drugs in excluded.items(): expanded.setdefault(d,set()).update(drugs)
    candidates={r['drug_id'] for r in read_csv(a.split/'drugs.csv')}
    if any(t not in candidates for _,t in conflict):
        raise ValueError('A conflict drug is outside fixed candidate list')
    rows=[]; detail=[]; graphs={}
    for label, path in [('0.3',a.split),('0.1',a.t01)]:
        diseases,sims,stats=build_similarity(path)
        graph,_=build_graph(diseases,sims,10)
        comms=communities_for(graph,1.5,42)
        graphs[label]={**stats,'edges':graph.number_of_edges(),'communities':len(comms)}
        for name,train in [('excluded',base),('included_in_training',expanded)]:
            ranked,native,_=make_rankings('louvain',truth,train,excluded,candidates,10,communities=comms,fallback=False)
            result,by_disease=metrics(ranked,truth,10,native)
            rows.append({'threshold':label,'conflict_policy':name,'train_pairs':sum(map(len,train.values())),**result})
            detail.extend({'threshold':label,'conflict_policy':name,**x} for x in by_disease)
            print(label,name,json.dumps(result),flush=True)
    write_csv(a.output/'metrics.csv',list(rows[0]),rows)
    write_csv(a.output/'per_disease.csv',list(detail[0]),detail)
    meta={'policy':'Same original split; append the 331 conflicts exclusively to train for included condition; always mask all 331 when recommending, to keep same fixed eligible test labels and mask.',
          'warning':'This isolates the effect of using conflicting edges as a training signal. It does not measure a test including conflicting labels, nor represent all 339 conflicts outside the selected disease scope.',
          'frozen_config':{'neighbors':10,'resolution':1.5,'seed':42},'conflicts':len(conflict),
          'test_diseases':len(truth),'test_pairs':sum(map(len,truth.values())),'graphs':graphs,'metrics':rows,
          'hashes':{str(a.split/name):checksum(a.split/name) for name in ('train.csv','test.csv','excluded.csv','disease_gene.csv')}}
    (a.output/'summary.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')

## 코드 6. 팀원 CBF 예비 분할 재현

CBF 노트북의 분할 규칙을 재현해 인기 약물과 Louvain을 평가한다. 이 셀은 함수만 정의한다.

In [ ]:
"""Reproduce CBF notebook's preliminary splits and evaluate strict Louvain.

Preliminary data contains contraindication-conflicting labels. Do not treat as a final protocol.
"""
import argparse
from collections import defaultdict
import csv
import hashlib
import json
from pathlib import Path
from statistics import mean, stdev
import time

import networkx as nx


def split_for_seed(pairs, seed):
    by_disease = group_pairs(pairs)
    result = {s: [] for s in ('train', 'validation', 'test')}
    for disease, drugs in sorted(by_disease.items()):
        def key(drug):
            digest = hashlib.blake2b(f'{seed}|{disease}|{drug}'.encode('utf-8'), digest_size=8).digest()
            return (int.from_bytes(digest, 'big'), drug)
        ordered = sorted(drugs, key=key)
        n = len(ordered)
        n_test = min(max(1, round(0.2*n)), n-2) if n >= 3 else 0
        n_val = min(max(1, round(0.2*n)), n-n_test-2) if n-n_test >= 3 else 0
        for i, drug in enumerate(ordered):
            part = 'test' if i < n_test else 'validation' if i < n_test + n_val else 'train'
            result[part].append({'disease_id': disease, 'drug_id': drug})
    return result


def ranking_metrics(rankings, truth, native):
    return {str(k): metrics(rankings, truth, k, native)[0] for k in (10, 100)}


def run_cbf_protocol():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--processed', type=Path, required=True)
    parser.add_argument('--data-output', type=Path, required=True)
    parser.add_argument('--results', type=Path, required=True)
    args = parser.parse_args()
    args.data_output.mkdir(parents=True, exist_ok=True)
    args.results.mkdir(parents=True, exist_ok=True)
    start = time.perf_counter()
    pairs = read_csv(args.processed/'indication.csv')
    pair_ids = {(r['disease_id'], r['drug_id']) for r in pairs}
    if len(pairs) != len(pair_ids) or len(pairs) != 58174:
        raise ValueError(f'Expected 58,174 unique original indications; found {len(pairs)} rows, {len(pair_ids)} pairs')
    diseases = sorted({d for d, _ in pair_ids})
    if len(diseases) != 2327: raise ValueError('Unexpected disease universe')
    write_csv(args.data_output/'diseases.csv', ['disease_id'], [{'disease_id': d} for d in diseases])
    # Use the same original overall thresholded data, retaining all indication diseases.
    gene_rows = read_csv(args.processed/'disease_gene_t03.csv')
    if len(gene_rows) != 810225: raise ValueError('Unexpected disease–gene input length')
    keep = set(diseases)
    genes = [r for r in gene_rows if r['disease_id'] in keep and float(r['evidence_score']) >= 0.3]
    write_csv(args.data_output/'disease_gene.csv', ['disease_id','gene_id'],
              [{'disease_id':r['disease_id'],'gene_id':r['gene_id']} for r in genes])
    del gene_rows, genes
    splits = {}
    for seed in range(42,47):
        parts = split_for_seed(pairs, seed)
        sets = {name: {(r['disease_id'],r['drug_id']) for r in rows} for name, rows in parts.items()}
        if len(sets['train']) != 35287 or len(sets['validation']) != 11350 or len(sets['test']) != 11537:
            raise ValueError(f'Unexpected split counts for seed {seed}: {[len(sets[n]) for n in sets]}')
        if sets['train'] & sets['validation'] or sets['train'] & sets['test'] or sets['validation'] & sets['test'] or set.union(*sets.values()) != pair_ids:
            raise ValueError('Split overlap or coverage error')
        if len({d for d, _ in sets['test']}) != 1662: raise ValueError('Unexpected test disease count')
        for part in parts:
            write_csv(args.data_output/f'{seed}_{part}.csv', ['disease_id','drug_id'], parts[part])
        splits[seed] = {name: group_pairs(rows) for name, rows in parts.items()}
    diseases, similarities, stats = build_similarity(args.data_output)
    val_rows, configs = [], {}
    for k in (10,20,30):
        graph, neighbors = build_graph(diseases, similarities, k)
        for resolution in (0.5,1.0,1.5):
            comms = communities_for(graph, resolution, 42)
            configs[(k,resolution)] = (graph, comms)
            scores=[]
            for seed, split in splits.items():
                train, truth = split['train'], split['validation']
                candidates = {drug for drugs in train.values() for drug in drugs}
                rankings,native,_ = make_rankings('louvain', truth, train, {}, candidates, 10,
                                                 communities=comms, fallback=False)
                measure,_ = metrics(rankings, truth, 10, native)
                row = {'seed':seed,'neighbors':k,'resolution':resolution,**measure}
                val_rows.append(row); scores.append(row)
            avg = mean(row['macro_ndcg'] for row in scores)
            print(f'validation neighbors={k} resolution={resolution} mean_ndcg10={avg:.6f}',flush=True)
    choices=[]
    for k,r in configs:
        sub=[x for x in val_rows if x['neighbors']==k and x['resolution']==r]
        choices.append({'neighbors':k,'resolution':r,'mean_ndcg10':mean(x['macro_ndcg'] for x in sub),
                        'mean_recall10':mean(x['macro_recall'] for x in sub)})
    choices.sort(key=lambda x:(-x['mean_ndcg10'],-x['mean_recall10'],x['neighbors'],x['resolution']))
    best=choices[0]
    graph, comms=configs[(best['neighbors'],best['resolution'])]
    test_rows=[]; disease_rows=[]
    for seed, split in splits.items():
        combined = {d: set(split['train'].get(d,set())) | set(split['validation'].get(d,set()))
                    for d in set(split['train']) | set(split['validation'])}
        candidates={drug for drugs in combined.values() for drug in drugs}
        for method in ('popularity','louvain'):
            rankings,native,_=make_rankings(method,split['test'],combined,{},candidates,100,
                                          communities=comms,fallback=False)
            for cutoff in (10,100):
                result,detail=metrics(rankings,split['test'],cutoff,native)
                test_rows.append({'seed':seed,'method':method,**result})
                if cutoff==10:
                    disease_rows.extend({'seed':seed,'method':method,**row} for row in detail)
            print('test',seed,method,json.dumps(ranking_metrics(rankings,split['test'],native)),flush=True)
    summary=[]
    for method in ('popularity','louvain'):
        for cutoff in (10,100):
            rows=[r for r in test_rows if r['method']==method and r['k']==cutoff]
            summary.append({'method':method,'k':cutoff,'n_splits':len(rows),
              'recall_mean':mean(r['macro_recall'] for r in rows),'recall_sd':stdev(r['macro_recall'] for r in rows),
              'ndcg_mean':mean(r['macro_ndcg'] for r in rows),'ndcg_sd':stdev(r['macro_ndcg'] for r in rows),
              'coverage_mean':mean(r['native_coverage'] for r in rows)})
    meta={'protocol':'CBF preliminary, unfiltered 58174 indications, NOT team final',
          'warning':'339 contraindication-conflicting positive links are retained to match the preliminary CBF results.',
          'seeds':[42,43,44,45,46],'test_diseases_each':1662,'validation_scheme':'mean of five preliminary validation splits',
          'validation_caution':'validation sets can overlap other seeds test sets; not independent held-out evaluation',
          'selected':best,'graph':{**stats,'edges':graph.number_of_edges(),'communities':len(comms),
                'isolated':len(list(nx.isolates(graph)))},'test':summary,'seconds':time.perf_counter()-start,
          'source_sha256':{x:checksum(args.processed/x) for x in ('indication.csv','disease_gene_t03.csv')},
          'split_sha256':{str(seed):{part:checksum(args.data_output/f'{seed}_{part}.csv') for part in ('train','validation','test')} for seed in splits}}
    write_csv(args.results/'validation_by_seed.csv',list(val_rows[0]),val_rows)
    write_csv(args.results/'validation_summary.csv',list(choices[0]),choices)
    write_csv(args.results/'test_by_seed.csv',list(test_rows[0]),test_rows)
    write_csv(args.results/'test_per_disease.csv',list(disease_rows[0]),disease_rows)
    (args.results/'summary.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
    print('DONE',json.dumps(meta,ensure_ascii=False),flush=True)

## 코드 7. 유전자 수별 분석

테스트 질병을 유전자 수로 나누어 결과를 보고하고, 점수 0.1과 0.3을 질병 단위로 비교한다. 이 셀은 함수만 정의한다.

In [ ]:
"""Report no-fallback test results by gene count and threshold on the same test set."""
import argparse
from collections import Counter
import csv
import json
from pathlib import Path
from statistics import median
import numpy as np


def read(path):
    with path.open(encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))


def bucket(n):
    if n == 0: return '0'
    if n == 1: return '1'
    if n < 5: return '2-4'
    if n < 10: return '5-9'
    return '10+'


def run_gene_count_analysis():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--t03-data', type=Path, required=True)
    parser.add_argument('--t01-data', type=Path, required=True)
    parser.add_argument('--results', type=Path, required=True)
    args = parser.parse_args()
    data = {'0.3': args.t03_data, '0.1': args.t01_data}
    genes = {t: Counter(r['disease_id'] for r in read(p / 'disease_gene.csv')) for t, p in data.items()}
    test = {t: {r['disease_id'] for r in read(p / 'test.csv')} for t, p in data.items()}
    if test['0.3'] != test['0.1']:
        raise ValueError('Test disease sets differ')
    for name in ['train.csv', 'test.csv', 'drugs.csv', 'excluded.csv', 'diseases.csv']:
        if (args.t03_data / name).read_bytes() != (args.t01_data / name).read_bytes():
            raise ValueError(f'Input files differ: {name}')
    result_rows = []
    results_by_threshold = {}
    for threshold in ['0.3', '0.1']:
        rows = {r['disease_id']: r for r in read(args.results / ('t' + threshold.replace('.', '')) / 'per_disease.csv')
                if r['method'] == 'louvain_no_fallback'}
        if set(rows) != test[threshold]:
            raise ValueError('Results and test diseases differ')
        results_by_threshold[threshold] = rows
        for grouping in ['threshold_specific', 'fixed_t03']:
            gene_counts = genes[threshold] if grouping == 'threshold_specific' else genes['0.3']
            for group in ['0','1','2-4','5-9','10+']:
                ids = [d for d in sorted(rows) if bucket(gene_counts[d]) == group]
                if not ids: continue
                hits = [rows[d] for d in ids if rows[d]['native_recommendation'] == 'True']
                result_rows.append({'threshold': threshold, 'grouping': grouping, 'gene_count': group,
                    'diseases': len(ids), 'native_count': len(hits), 'native_coverage': len(hits)/len(ids),
                    'macro_recall_all': sum(float(rows[d]['recall']) for d in ids)/len(ids),
                    'macro_ndcg_all': sum(float(rows[d]['ndcg']) for d in ids)/len(ids),
                    'macro_recall_native': sum(float(r['recall']) for r in hits)/len(hits) if hits else None,
                    'macro_ndcg_native': sum(float(r['ndcg']) for r in hits)/len(hits) if hits else None})
        print(json.dumps({'threshold':threshold, 'unique_pairs':sum(genes[threshold].values()),
                          'overall_diseases_with_genes':len(genes[threshold]),
                          'eval_without_genes':sum(genes[threshold][d]==0 for d in test[threshold]),
                          'eval_median_genes':median(genes[threshold][d] for d in test[threshold])}), flush=True)
    out = args.results / 'gene_count_analysis.csv'
    with out.open('w', encoding='utf-8', newline='') as f:
        writer=csv.DictWriter(f, fieldnames=list(result_rows[0]));writer.writeheader();writer.writerows(result_rows)
    ids = sorted(test['0.3'])
    rng = np.random.default_rng(42)
    differences = {}
    for metric in ('recall', 'ndcg'):
        d = np.array([float(results_by_threshold['0.1'][i][metric]) -
                      float(results_by_threshold['0.3'][i][metric]) for i in ids])
        samples = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(5000)])
        differences[metric] = {'difference_t01_minus_t03': float(d.mean()),
            'bootstrap_95_ci': [float(x) for x in np.quantile(samples, [0.025, 0.975])],
            'improved': int((d > 1e-12).sum()), 'worsened': int((d < -1e-12).sum()),
            'unchanged': int((np.abs(d) <= 1e-12).sum())}
    (args.results / 'paired_threshold_comparison.json').write_text(
        json.dumps({'seed': 42, 'resamples': 5000, 'unit': 'disease', 'metrics': differences}, indent=2)+'\n', encoding='utf-8')
    print(json.dumps(result_rows, ensure_ascii=False, indent=2), flush=True)

## 코드 8. 유사도 가중 (개선)

같은 그룹 안에서 유사도를 가중한 약물 점수를 기존 방식과 비교한다. 이 셀은 함수만 정의한다.

In [ ]:
"""Compare uniform and similarity-weighted community votes on fixed preliminary splits."""
import argparse
from collections import defaultdict
import json
from pathlib import Path
from statistics import mean, stdev
import numpy as np
from scipy.sparse import coo_matrix, diags


def weighted_cosine_similarity(processed, diseases):
    index = {d:i for i,d in enumerate(diseases)}
    edges = [r for r in read_csv(processed/'disease_gene_t03.csv')
             if r['disease_id'] in index and float(r['evidence_score']) >= 0.3]
    pairs = {(r['disease_id'],r['gene_id']) for r in edges}
    if len(pairs) != len(edges): raise ValueError('Duplicate gene relations')
    genes = {g:i for i,g in enumerate(sorted({r['gene_id'] for r in edges}))}
    rr = np.array([index[r['disease_id']] for r in edges],dtype=np.int32)
    cc = np.array([genes[r['gene_id']] for r in edges],dtype=np.int32)
    values = np.array([float(r['evidence_score']) for r in edges])
    x = coo_matrix((values,(rr,cc)),shape=(len(diseases),len(genes))).tocsr()
    norms = np.sqrt(np.asarray(x.multiply(x).sum(axis=1)).ravel())
    inv = np.divide(1.0,norms,out=np.zeros_like(norms),where=norms>0)
    normalized=diags(inv) @ x
    s=(normalized @ normalized.T).tocsr();s.setdiag(0);s.eliminate_zeros()
    rows=[]
    for i in range(len(diseases)):
        row=s.getrow(i)
        keep=row.data>0; idx=row.indices[keep]; weight=row.data[keep]
        order=np.lexsort((idx,-weight))
        rows.append([(diseases[int(j)],float(w)) for j,w in zip(idx[order],weight[order])])
    return rows


def similarity_weighted_rankings(queries, train, candidates, communities, similarities, top_n=100):
    """Score a drug by sum(sim(query, other)) over other community diseases with the drug.

    All community members participate when similarity > 0; absent/zero-similarity members
    contribute zero. Query disease and its known drugs are excluded.
    """
    community={d:i for i, group in enumerate(communities) for d in group}
    rankings={}; native={}
    for disease in sorted(queries):
        known=train.get(disease,set()); scores=defaultdict(float)
        if disease in community:
            group=communities[community[disease]]
            for other in sorted(group):
                if other==disease: continue
                weight=similarities.get(disease,{}).get(other,0.0)
                if weight<=0: continue
                for drug in train.get(other,set()):
                    if drug in candidates and drug not in known: scores[drug]+=weight
        ranked=sorted(scores,key=lambda d:(-scores[d],d))[:top_n]
        rankings[disease]=ranked; native[disease]=bool(ranked)
    return rankings,native


def run_improved_louvain():
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--processed',type=Path,required=True)
    parser.add_argument('--split-data',type=Path,required=True)
    parser.add_argument('--output',type=Path,required=True)
    a=parser.parse_args();a.output.mkdir(parents=True,exist_ok=True)
    diseases,jaccard,stats=build_similarity(a.split_data)
    cosine=weighted_cosine_similarity(a.processed,diseases)
    # All variants use the previously selected graph neighbourhood and resolution; no test tuning.
    graph_j,_=build_graph(diseases,jaccard,10); comm_j=communities_for(graph_j,1.5,42)
    graph_c,_=build_graph(diseases,cosine,10); comm_c=communities_for(graph_c,1.5,42)
    jaccard_map={d:dict(row) for d,row in zip(diseases,jaccard)}
    cosine_map={d:dict(row) for d,row in zip(diseases,cosine)}
    configurations=[('original_jaccard_uniform',jaccard_map,comm_j,False),
                    ('jaccard_similarity_weighted',jaccard_map,comm_j,True),
                    ('score_cosine_similarity_weighted',cosine_map,comm_c,True)]
    rows=[];details=[]
    for seed in range(42,47):
        parts={name:group_pairs(read_csv(a.split_data/f'{seed}_{name}.csv')) for name in ('train','validation','test')}
        for phase in ('validation','test'):
            train=parts['train'] if phase=='validation' else {
              d:parts['train'].get(d,set())|parts['validation'].get(d,set())
              for d in set(parts['train'])|set(parts['validation'])}
            truth=parts[phase]; candidates={drug for v in train.values() for drug in v}
            for name,sim,comms,weighted in configurations:
                if weighted:
                    ranking,native=similarity_weighted_rankings(truth,train,candidates,comms,sim)
                else:
                    ranking,native,_=make_rankings('louvain',truth,train,{},candidates,100,communities=comms,fallback=False)
                for k in (10,100):
                    result,per=metrics(ranking,truth,k,native)
                    rows.append({'phase':phase,'seed':seed,'method':name,**result})
                    if phase=='test' and k==10:
                        details.extend({'seed':seed,'method':name,**r} for r in per)
                print(phase,seed,name,'recall10',rows[-2]['macro_recall'],'ndcg10',rows[-2]['macro_ndcg'],flush=True)
    summary=[]
    for phase in ('validation','test'):
        for name,*_ in configurations:
            for k in (10,100):
                sub=[r for r in rows if r['phase']==phase and r['method']==name and r['k']==k]
                summary.append({'phase':phase,'method':name,'k':k,
                    'recall_mean':mean(r['macro_recall'] for r in sub),'recall_sd':stdev(r['macro_recall'] for r in sub),
                    'ndcg_mean':mean(r['macro_ndcg'] for r in sub),'ndcg_sd':stdev(r['macro_ndcg'] for r in sub),
                    'coverage_mean':mean(r['native_coverage'] for r in sub)})
    meta={'protocol':'CBF preliminary split 42-46; unfiltered labels contain 339 contraindication conflicts',
          'test_caution':'exploratory comparison on already-used overlapping tests; no independent final performance claim',
          'parameters':{'neighbors':10,'resolution':1.5,'louvain_seed':42,'gene_threshold':0.3,'score_power':1},
          'graph':{'original_jaccard':{'edges':graph_j.number_of_edges(),'communities':len(comm_j)},
                   'score_cosine':{'edges':graph_c.number_of_edges(),'communities':len(comm_c)}},
          'input_sha256':{'disease_gene_t03.csv':checksum(a.processed/'disease_gene_t03.csv'),
                           **{f'{seed}_{part}.csv':checksum(a.split_data/f'{seed}_{part}.csv') for seed in range(42,47) for part in ('train','validation','test')}},
          'summary':summary}
    write_csv(a.output/'metrics_by_seed.csv',list(rows[0]),rows)
    write_csv(a.output/'per_disease.csv',list(details[0]),details)
    write_csv(a.output/'summary.csv',list(summary[0]),summary)
    (a.output/'summary.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
    print('DONE',json.dumps(summary),flush=True)

## 코드 9. 특징 유사도와 점수 계산 함수

연관 점수 또는 `연관 점수 × IDF`의 코사인 유사도, 직접 이웃 가중치, 검증 점수 계산 함수를 정의한다. 코드 10과 11에서 쓴다.

In [ ]:
"""Shared helpers for the group-boundary analyses: score or IDF cosine similarity,
direct-neighbor weights, and validation scoring.
"""
import argparse
from collections import defaultdict
from pathlib import Path
import json
from statistics import mean, stdev
import numpy as np
from scipy.sparse import coo_matrix, csr_matrix, diags


def feature_similarities(processed, diseases):
    index={d:i for i,d in enumerate(diseases)}
    edges=read_csv(processed/'disease_gene_t03.csv')
    if len(edges)!=810225: raise ValueError('Unexpected original feature data')
    # Match the CBF notebook's IDF universe: all diseases in the feature file.
    all_diseases={r['disease_id'] for r in edges}
    gene_disease=defaultdict(set)
    for r in edges: gene_disease[r['gene_id']].add(r['disease_id'])
    filtered=[r for r in edges if r['disease_id'] in index and float(r['evidence_score'])>=0.3]
    if len({(r['disease_id'],r['gene_id']) for r in filtered})!=len(filtered):
        raise ValueError('Duplicated disease-gene relationships')
    genes=sorted({r['gene_id'] for r in filtered}); gindex={g:i for i,g in enumerate(genes)}
    rr=np.array([index[r['disease_id']] for r in filtered],dtype=np.int32)
    cc=np.array([gindex[r['gene_id']] for r in filtered],dtype=np.int32)
    val=np.array([float(r['evidence_score']) for r in filtered],dtype=np.float64)
    x=coo_matrix((val,(rr,cc)),shape=(len(diseases),len(genes))).tocsr()
    idf=np.array([np.log((len(all_diseases)+1)/(len(gene_disease[g])+1))+1 for g in genes])
    sims={}
    for mode,mat in [('score_cosine',x),('idf_cosine',x.multiply(idf).tocsr())]:
        norm=np.sqrt(np.asarray(mat.multiply(mat).sum(axis=1)).ravel())
        inv=np.divide(1.0,norm,out=np.zeros_like(norm),where=norm>0)
        unit=diags(inv)@mat
        similarity=(unit@unit.T).toarray().astype(np.float32)
        np.fill_diagonal(similarity,0)
        sims[mode]=similarity
    return sims


def sorted_similarity_rows(diseases, matrix):
    ids=np.arange(len(diseases)); rows=[]
    for row in matrix:
        indices=ids[row>0]; values=row[indices]
        order=np.lexsort((indices,-values))
        rows.append([(diseases[int(i)],float(v)) for i,v in zip(indices[order],values[order])])
    return rows


def train_matrix(train,diseases):
    ids={d:i for i,d in enumerate(diseases)}
    drugs=sorted({x for values in train.values() for x in values}); drug_ix={d:i for i,d in enumerate(drugs)}
    pairs=[(ids[d],drug_ix[t]) for d,values in train.items() if d in ids for t in values]
    matrix=coo_matrix((np.ones(len(pairs)),([x[0] for x in pairs],[x[1] for x in pairs])),
                      shape=(len(diseases),len(drugs))).tocsr()
    return drugs,drug_ix,matrix


def score_from_weights(weights, train_features, train, truth, diseases, drugs, drug_ix, cutoff=10):
    query_index={d:i for i,d in enumerate(diseases)}
    # Compute only queries with validation positives, keeping zero-score queries.
    query_ids=sorted(truth); indices=np.array([query_index[d] for d in query_ids])
    scored=(weights[indices]@train_features).tocsr(); rankings={};native={}
    for i,d in enumerate(query_ids):
        row=scored.getrow(i); ids=row.indices; data=row.data
        forbidden={drug_ix[x] for x in train.get(d,set()) if x in drug_ix}
        eligible=np.array([(int(j) not in forbidden) and (float(s)>0) for j,s in zip(ids,data)],dtype=bool)
        ids=ids[eligible];data=data[eligible]
        order=np.lexsort((ids,-data))[:cutoff]
        rankings[d]=[drugs[int(j)] for j in ids[order]];native[d]=bool(len(order))
    return metrics(rankings,truth,cutoff,native)[0]


def direct_weights(sim,source_mask,neighbors,power):
    out=np.zeros_like(sim,dtype=np.float32)
    sources=np.flatnonzero(source_mask)
    for i,row in enumerate(sim):
        valid=sources[row[sources]>0]
        if not len(valid):continue
        order=np.lexsort((valid,-row[valid]))[:neighbors]; sel=valid[order]
        out[i,sel]=row[sel] if power==1 else row[sel]**2
    return csr_matrix(out)


## 코드 10. 그룹 경계와 이웃 수 비교

그룹 안 이웃, 직접 이웃, 그룹 밖 이웃 추가, 인기 약물 감점을 검증 분할에서 비교한다. 이 셀은 함수만 정의한다.

In [ ]:
"""Validation-only ablations of community boundaries, top-K neighbors, and popularity."""
import argparse
from collections import defaultdict
import json
from pathlib import Path
from statistics import mean, stdev
import numpy as np
from scipy.sparse import csr_matrix, diags
def group_membership(diseases, communities):
    ix={d:i for i,d in enumerate(diseases)};out=np.empty(len(diseases),dtype=np.int32)
    for group, members in enumerate(communities):
        for d in members:out[ix[d]]=group
    return out


def ranked_neighbors(sim, group, source_mask, k, direction, power=2):
    """Restrict to same or outside community, then select top K train diseases."""
    mat=np.zeros_like(sim,dtype=np.float32);sources=np.flatnonzero(source_mask)
    for i,row in enumerate(sim):
        group_mask=(group[sources]==group[i])
        valid=sources[(row[sources]>0) & (group_mask if direction=='inside' else ~group_mask)]
        order=np.lexsort((valid,-row[valid]))[:k];indices=valid[order]
        mat[i,indices]=row[indices]**power
    return csr_matrix(mat)


def compute_case(seed, sim, group, diseases, train, truth, processed):
    """Find a validation disease where a direct top-10 hit is missed by group top-10."""
    disease_index={d:i for i,d in enumerate(diseases)}
    sources=np.array([i for i,d in enumerate(diseases) if train.get(d)],dtype=np.int32)
    disease_names={r['disease_id']:r.get('name','') for r in read_csv(processed/'disease_names.csv')}
    drug_names={r['drug_id']:r.get('name','') for r in read_csv(processed/'drug_names.csv')}
    def rank(d,inside):
        i=disease_index[d]; valid=sources[sim[i,sources]>0]
        if inside:valid=valid[group[valid]==group[i]]
        chosen=valid[np.lexsort((valid,-sim[i,valid]))[:50]]
        scores=defaultdict(float); contributions=defaultdict(list)
        for j in chosen:
            other=diseases[int(j)];w=float(sim[i,j])**2
            for drug in train.get(other,set())-train.get(d,set()):
                scores[drug]+=w;contributions[drug].append((other,float(sim[i,j]),bool(group[j]==group[i])))
        order=sorted(scores,key=lambda drug:(-scores[drug],drug))
        return order,contributions
    for disease in sorted(truth):
        direct,dc=rank(disease,False);inside,ic=rank(disease,True)
        missing=(set(direct[:10]) & truth[disease])-set(inside[:10])
        for drug in sorted(missing):
            outside=sorted((x for x in dc[drug] if not x[2]),key=lambda x:(-x[1],x[0]))
            if outside:
                other,simvalue,_=outside[0]
                return {'seed':seed,'disease_id':disease,'disease_name':disease_names.get(disease,''),
                        'test_status':'validation held-out association, not efficacy',
                        'hidden_drug_id':drug,'hidden_drug_name':drug_names.get(drug,''),
                        'direct_rank':direct.index(drug)+1,'within_group_rank':inside.index(drug)+1 if drug in inside else None,
                        'outside_contributor_id':other,'outside_contributor_name':disease_names.get(other,''),
                        'outside_contributor_similarity':simvalue,'drug_contributor_count_direct':len(dc[drug]),
                        'qualification':'Example selected after evaluation; illustrates one case, not a causal or clinical claim.'}
    return None


def run_community_diagnostics():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument('--processed',type=Path,required=True);p.add_argument('--split-data',type=Path,required=True)
    p.add_argument('--output',type=Path,required=True);a=p.parse_args();a.output.mkdir(parents=True,exist_ok=True)
    diseases=sorted(r['disease_id'] for r in read_csv(a.split_data/'diseases.csv'))
    sim=feature_similarities(a.processed,diseases)['idf_cosine']
    graph,_=build_graph(diseases,sorted_similarity_rows(diseases,sim),10)
    comms=communities_for(graph,1.0,42);group=group_membership(diseases,comms)
    rows=[];case=None
    for seed in range(42,47):
        train=group_pairs(read_csv(a.split_data/f'{seed}_train.csv'))
        truth=group_pairs(read_csv(a.split_data/f'{seed}_validation.csv'))
        drugs,drug_ix,R=train_matrix(train,diseases)
        sources=np.array([bool(train.get(d)) for d in diseases]);ident=diags(np.ones(len(drugs)),format='csr')
        for k in (10,20,50):
            for method in ('group_top_k','direct_top_k'):
                W=(ranked_neighbors(sim,group,sources,k,'inside') if method=='group_top_k'
                   else direct_weights(sim,sources,k,2))
                result=score_from_weights(W,R,train,truth,diseases,drugs,drug_ix)
                rows.append({'seed':seed,'method':method,'neighbor_k':k,'outside_k':0,'outside_weight':0.,'popularity_beta':0.,**result})
        inside=ranked_neighbors(sim,group,sources,50,'inside')
        for outside_k in (10,20):
            outside=ranked_neighbors(sim,group,sources,outside_k,'outside')
            for weight in (.25,.5):
                result=score_from_weights(inside+weight*outside,R,train,truth,diseases,drugs,drug_ix)
                rows.append({'seed':seed,'method':'outside_added','neighbor_k':50,'outside_k':outside_k,'outside_weight':weight,'popularity_beta':0.,**result})
        # A modest global-popularity penalty, estimated from the train split only.
        freq=np.asarray(R.sum(axis=0)).ravel()
        adjusted=(inside@R)@diags((freq+1.)**(-.5),format='csr')
        result=score_from_weights(adjusted,ident,train,truth,diseases,drugs,drug_ix)
        rows.append({'seed':seed,'method':'group_top_k_popularity_penalty','neighbor_k':50,'outside_k':0,'outside_weight':0.,'popularity_beta':.5,**result})
        if seed==42:case=compute_case(seed,sim,group,diseases,train,truth,a.processed)
        print('VALIDATION seed',seed,'complete',flush=True)
    keys=('method','neighbor_k','outside_k','outside_weight','popularity_beta');summary=[]
    for key in sorted({tuple(r[n] for n in keys) for r in rows}):
        sub=[r for r in rows if tuple(r[n] for n in keys)==key]
        if len(sub)!=5 or len({r['seed'] for r in sub})!=5: raise ValueError('Expected one result per seed and configuration')
        summary.append({**dict(zip(keys,key)),'n_splits':len(sub),'mean_recall10':mean(x['macro_recall'] for x in sub),
            'sd_recall10':stdev(x['macro_recall'] for x in sub),'mean_ndcg10':mean(x['macro_ndcg'] for x in sub),
            'sd_ndcg10':stdev(x['macro_ndcg'] for x in sub),'mean_coverage':mean(x['native_coverage'] for x in sub)})
    summary.sort(key=lambda r:(-r['mean_ndcg10'],-r['mean_recall10'],str(r)))
    meta={'scope':'validation only; same 5 preliminary train/validation splits; no test labels loaded',
          'limitation':'339 conflicting labels retained, folds overlap prior observed tests; exploration, not final holdout',
          'feature':'score times IDF cosine','similarity_power':2,'graph_neighbors':10,'resolution':1.0,'louvain_seed':42,
          'outside_rule':'sum(top 50 within-community scores) plus weight times top outside-K scores; no normalization',
          'input_sha256':{'disease_gene_t03':checksum(a.processed/'disease_gene_t03.csv'),
            **{f'{s}_{kind}':checksum(a.split_data/f'{s}_{kind}.csv') for s in range(42,47) for kind in ('train','validation')}},
          'case':case,'summary':summary}
    write_csv(a.output/'by_seed.csv',list(rows[0]),rows);write_csv(a.output/'summary.csv',list(summary[0]),summary)
    (a.output/'summary.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
    print('CASE',json.dumps(case,ensure_ascii=False),flush=True)
    print('DONE',json.dumps(summary,ensure_ascii=False),flush=True)

## 코드 11. 비슷한 질병의 그룹 내 보존 비율

질병마다 가장 비슷한 학습 질병 50개 중 같은 그룹에 있는 비율을 구한다. 이 셀은 함수만 정의한다.

In [ ]:
"""Explain which nearest-neighbor relationships survive Louvain's hard boundary.

Development-only: reads train and validation labels, never the test split.
"""
import argparse
import json
from pathlib import Path
from statistics import mean,median
import numpy as np


def rankings_for(weights,R,train,truth,diseases,drugs,drug_ix):
    idx={d:i for i,d in enumerate(diseases)};queries=sorted(truth)
    scores=(weights[[idx[d] for d in queries]]@R).tocsr();ranking={};native={}
    for i,d in enumerate(queries):
        row=scores.getrow(i); ids=row.indices; values=row.data
        forbidden={drug_ix[x] for x in train.get(d,set()) if x in drug_ix}
        valid=np.array([(int(j) not in forbidden and float(v)>0) for j,v in zip(ids,values)],dtype=bool)
        ids=ids[valid];values=values[valid];order=np.lexsort((ids,-values))[:10]
        ranking[d]=[drugs[int(j)] for j in ids[order]];native[d]=bool(len(order))
    return ranking,native


def run_community_retention():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument('--processed',type=Path,required=True);p.add_argument('--split-data',type=Path,required=True)
    p.add_argument('--output',type=Path,required=True);a=p.parse_args();a.output.mkdir(parents=True,exist_ok=True)
    diseases=sorted(r['disease_id'] for r in read_csv(a.split_data/'diseases.csv'))
    sim=feature_similarities(a.processed,diseases)['idf_cosine']
    graph,_=build_graph(diseases,sorted_similarity_rows(diseases,sim),10)
    comms=communities_for(graph,1.0,42);group=group_membership(diseases,comms)
    disease_ix={d:i for i,d in enumerate(diseases)};details=[];summaries=[]
    for seed in range(42,47):
        train=group_pairs(read_csv(a.split_data/f'{seed}_train.csv'))
        truth=group_pairs(read_csv(a.split_data/f'{seed}_validation.csv'))
        drugs,drug_ix,R=train_matrix(train,diseases)
        sources=np.array([bool(train.get(d)) for d in diseases]);source_ids=np.flatnonzero(sources)
        inside=ranked_neighbors(sim,group,sources,50,'inside')
        direct=direct_weights(sim,sources,50,2)
        in_rank,in_native=rankings_for(inside,R,train,truth,diseases,drugs,drug_ix)
        di_rank,di_native=rankings_for(direct,R,train,truth,diseases,drugs,drug_ix)
        _,in_metrics=metrics(in_rank,truth,10,in_native); _,di_metrics=metrics(di_rank,truth,10,di_native)
        in_m={r['disease_id']:r for r in in_metrics};di_m={r['disease_id']:r for r in di_metrics}
        for d in sorted(truth):
            i=disease_ix[d];valid=source_ids[sim[i,source_ids]>0]
            nearest=valid[np.lexsort((valid,-sim[i,valid]))[:50]]
            retained=int(np.sum(group[nearest]==group[i]));outside=len(nearest)-retained
            details.append({'seed':seed,'disease_id':d,'top50_available':len(nearest),
                'top50_inside':retained,'top50_outside':outside,
                'retained_fraction':retained/len(nearest) if len(nearest) else None,
                'community_size':len(comms[group[i]]),'validation_positives':len(truth[d]),
                'group_recall10':in_m[d]['recall'],'direct_recall10':di_m[d]['recall'],
                'group_ndcg10':in_m[d]['ndcg'],'direct_ndcg10':di_m[d]['ndcg'],
                'group_hits':in_m[d]['hits'],'direct_hits':di_m[d]['hits'],
                'direct_only_hits':len((set(di_rank[d][:10])-set(in_rank[d][:10]))&truth[d]),
                'group_only_hits':len((set(in_rank[d][:10])-set(di_rank[d][:10]))&truth[d])})
        sub=[r for r in details if r['seed']==seed];valid=[r for r in sub if r['top50_available']>0]
        summaries.append({'seed':seed,'validation_diseases':len(sub),'diseases_with_neighbors':len(valid),
            'no_neighbors':len(sub)-len(valid),'mean_retained_fraction':mean(r['retained_fraction'] for r in valid),
            'median_retained_fraction':median(r['retained_fraction'] for r in valid),
            'mean_top50_inside':mean(r['top50_inside'] for r in valid),
            'mean_top50_outside':mean(r['top50_outside'] for r in valid),
            'some_outside':sum(r['top50_outside']>0 for r in valid),
            'median_community_size':median(r['community_size'] for r in valid),
            'direct_recall_wins':sum(r['direct_recall10']>r['group_recall10']+1e-12 for r in sub),
            'group_recall_wins':sum(r['group_recall10']>r['direct_recall10']+1e-12 for r in sub),
            'recall_ties':sum(abs(r['direct_recall10']-r['group_recall10'])<=1e-12 for r in sub),
            'direct_only_hits':sum(r['direct_only_hits'] for r in sub),
            'group_only_hits':sum(r['group_only_hits'] for r in sub)})
        print('VALIDATION SEED',seed,json.dumps(summaries[-1]),flush=True)
    write_csv(a.output/'by_disease.csv',list(details[0]),details)
    write_csv(a.output/'by_seed.csv',list(summaries[0]),summaries)
    report={'scope':'validation only; no test file read','seeds':[42,43,44,45,46],
      'settings':{'feature':'evidence score x IDF cosine','neighbor_k':50,'power':2,'graph_neighbors':10,'resolution':1.0,'seed':42},
      'retention_definition':'Among up to 50 most similar train diseases (positive similarity), fraction in the query disease Louvain community. Mean excludes diseases with no neighbors.',
      'comparison_definition':'Per validation disease compare top-10 recall using top-50 group versus top-50 direct; equal recall can conceal different predictions; not causal proof.',
      'caution':'Samples are descriptive; overlapping random splits, 339 conflicting labels and evidence leakage risk; post-hoc analysis.',
      'graph':{'diseases':len(diseases),'communities':len(comms),'edges':graph.number_of_edges(),
               'isolated':sum(len(c)==1 for c in comms)},
      'input_sha256':{'disease_gene_t03':checksum(a.processed/'disease_gene_t03.csv'),
          **{f'{s}_{kind}':checksum(a.split_data/f'{s}_{kind}.csv') for s in range(42,47) for kind in ('train','validation')}},
      'per_seed':summaries}
    (a.output/'summary.json').write_text(json.dumps(report,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')

## 코드 12. 그룹 밖 비율과 성능 차이

코드 11이 만든 질병별 결과(`by_disease.csv`)로, 그룹 안에 남은 비율 구간마다 그룹 안 이웃과 직접 이웃의 NDCG@10 차이를 구한다. 이 셀은 함수만 정의한다.

In [ ]:
"""Compare group-only and direct-neighbor NDCG@10 by the share of similar diseases kept in the group.

Reads by_disease.csv written by run_community_retention (validation only, no test labels).
"""
import csv
import numpy as np
from scipy.stats import spearmanr


def run_retention_vs_gap(by_disease_csv, seed=42):
    with open(by_disease_csv, encoding="utf-8", newline="") as handle:
        rows = [r for r in csv.DictReader(handle)
                if r["seed"] == str(seed) and r["top50_available"] and int(r["top50_available"]) > 0]
    pick = lambda key: np.array([float(r[key]) for r in rows])
    kept, size = pick("retained_fraction"), pick("community_size")
    group, direct = pick("group_ndcg10"), pick("direct_ndcg10")
    gap = direct - group
    print(f"seed {seed}: {len(rows)} diseases with similar training diseases")
    print(f"mean NDCG@10 group {group.mean():.4f}, direct {direct.mean():.4f}")
    edges = [0, .2, .4, .6, .8, 1.01]
    for low, high in zip(edges[:-1], edges[1:]):
        m = (kept >= low) & (kept < high)
        print(f"kept [{low:.1f},{high:.1f}) n={m.sum():4d} group={group[m].mean():.4f} "
              f"direct={direct[m].mean():.4f} gap={gap[m].mean():+.4f}")
    print("spearman(kept, gap) = %.3f" % spearmanr(kept, gap)[0])
    print("spearman(community size, gap) = %.3f" % spearmanr(size, gap)[0])
    won_g, won_d = gap < -1e-12, gap > 1e-12
    print(f"group better {won_g.sum()} (mean kept {kept[won_g].mean():.2f}), "
          f"direct better {won_d.sum()} (mean kept {kept[won_d].mean():.2f})")

## 전체 재실행

공유 분할, 원본 parquet, processed CSV를 읽어 별도 `WORK_DIR`에 결과를 만든다. CBF 예비 분할은 정제 전 적응증 58,174쌍을 쓰며 팀 최종 분할이 아니다. 기존 결과 파일은 건드리지 않는다. 실행 뒤 `summary.json`, `metrics.csv` 등을 확인한다.

In [ ]:
def call_cli(function, arguments):
    previous = sys.argv[:]
    try:
        sys.argv = [function.__name__, *[str(x) for x in arguments]]
        function()
    finally:
        sys.argv = previous

if RUN_EXPERIMENTS:
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    t01 = WORK_DIR / "data" / "t01"
    call_cli(prepare_threshold_data, ["--raw", RAW_PARQUET, "--split", SPLIT_DIR,
                                     "--output", t01, "--threshold", "0.1"])
    call_cli(run_conflict_ablation, ["--split", SPLIT_DIR, "--t01", t01,
             "--indication", PROCESSED_DIR / "indication.csv", "--output", WORK_DIR / "conflict_ablation"])
    for label, data in [("t03", SPLIT_DIR), ("t01", t01)]:
        call_cli(run_original_experiment, ["--data-dir", data, "--output-dir", WORK_DIR / "gene_analysis" / label])
    call_cli(run_gene_count_analysis, ["--t03-data", SPLIT_DIR, "--t01-data", t01,
                                      "--results", WORK_DIR / "gene_analysis"])
    call_cli(run_cbf_protocol, ["--processed", PROCESSED_DIR, "--data-output", WORK_DIR / "cbf_data",
                                "--results", WORK_DIR / "cbf_results"])
    call_cli(run_improved_louvain, ["--processed", PROCESSED_DIR, "--split-data", WORK_DIR / "cbf_data",
                                    "--output", WORK_DIR / "improved_results"])
    call_cli(run_community_diagnostics, ["--processed", PROCESSED_DIR, "--split-data", WORK_DIR / "cbf_data",
                                         "--output", WORK_DIR / "community_diagnostics"])
    call_cli(run_community_retention, ["--processed", PROCESSED_DIR, "--split-data", WORK_DIR / "cbf_data",
                                       "--output", WORK_DIR / "community_retention"])
    run_retention_vs_gap(WORK_DIR / "community_retention" / "by_disease.csv")
    print("저장 위치:", WORK_DIR)
else:
    print("재실행하지 않았다. 위에 저장된 실험 결과를 확인한다.")